In [1]:
import os
import gc
import pandas as pd
import time
from datetime import datetime, timedelta
from pathlib import Path
from tqdm import tqdm
from langchain_openai import ChatOpenAI
import re
import inspect
import types
import numpy as np
import logging as log
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [2]:
def to_lc_messages(messages_payload):
    lc_messages = []
    for m in messages_payload:
        role = m.get("role")
        content = m.get("content", "")
        if role == "system":
            lc_messages.append(SystemMessage(content=content))
        elif role == "user":
            lc_messages.append(HumanMessage(content=content))
        elif role == "assistant":
            lc_messages.append(AIMessage(content=content))
        else:
            lc_messages.append(HumanMessage(content=content))
    return lc_messages


#### 全局配置区

In [3]:
# ==========================================
# 0. 全局配置：定义“季频快照”日历
# ==========================================
def generate_snapshot_calendar(start_year, end_year):
    dates = []
    for year in range(start_year, end_year + 1):
        # 每年只有这 3 个统一更新日
        dates.append(f"{year}-04-30") # 年报 + 一季报
        dates.append(f"{year}-08-30") # 中报
        dates.append(f"{year}-10-30") # 三季报
    return pd.to_datetime(dates).sort_values()

START_YEAR = 2016
END_YEAR = 2025

# 🔥 核心修改：全局日历变更为稀疏的快照日
GLOBAL_TRADING_DAYS = generate_snapshot_calendar(START_YEAR, END_YEAR)

print(f"📅 [Calendar] 季频快照日历已构建: {len(GLOBAL_TRADING_DAYS)} 个切片日")
print(f"   样例: {GLOBAL_TRADING_DAYS[:6].strftime('%Y-%m-%d').tolist()} ...")

📅 [Calendar] 季频快照日历已构建: 30 个切片日
   样例: ['2016-04-30', '2016-08-30', '2016-10-30', '2017-04-30', '2017-08-30', '2017-10-30'] ...


In [4]:
GLOBAL_TRADING_DAYS

DatetimeIndex(['2016-04-30', '2016-08-30', '2016-10-30', '2017-04-30',
               '2017-08-30', '2017-10-30', '2018-04-30', '2018-08-30',
               '2018-10-30', '2019-04-30', '2019-08-30', '2019-10-30',
               '2020-04-30', '2020-08-30', '2020-10-30', '2021-04-30',
               '2021-08-30', '2021-10-30', '2022-04-30', '2022-08-30',
               '2022-10-30', '2023-04-30', '2023-08-30', '2023-10-30',
               '2024-04-30', '2024-08-30', '2024-10-30', '2025-04-30',
               '2025-08-30', '2025-10-30'],
              dtype='datetime64[ns]', freq=None)

#### 历史记忆

In [5]:
import re
from pathlib import Path

CHN_RANGE = r"\u4e00-\u9fff"

def compress_factor_table(md_text: str) -> str:
    """
    从 LLM 返回的 markdown 因子表格中提取【因子名称 + 计算方式】两列，
    以纯文本形式返回，用于历史记忆瘦身。
    行格式示例：
        Volatility-Adjusted Momentum: VAM_t = ...
    """
    lines = md_text.splitlines()
    rows = []

    for line in lines:
        line = line.strip()
        if not line.startswith("|"):
            continue

        # 去掉两侧竖线
        inner = line.strip("|").strip()

        # 跳过分隔线行：比如 '---------|---------|---------'、':---|:---|' 等
        inner_no_pipes = inner.replace("|", "").strip()
        if inner_no_pipes and set(inner_no_pipes) <= set("-:"):
            continue

        # 用“| + 中文”确定【公式列】和【解释说明】之间的那根竖线（取最后一个）
        cn_bar_match = None
        pattern = rf"\|[ \t]*[{CHN_RANGE}]"
        for m in re.finditer(pattern, inner):
            cn_bar_match = m

        # 如果这一行根本没有“| + 中文”，大概率不是标准的因子行，直接跳过
        if not cn_bar_match:
            continue

        bar_idx = cn_bar_match.start()   # 竖线所在的位置
        left = inner[:bar_idx].rstrip()  # 左侧：名称 + 公式两列
        # right = inner[bar_idx+1:].lstrip()  # 右侧是中文解释，你要丢掉就不用管

        # 再在左侧拆一次：只拆第一个“未被反斜杠转义”的 '|'
        parts = [c.strip() for c in re.split(r'(?<!\\)\|', left, maxsplit=1)]
        if len(parts) < 2:
            continue

        name, formula = parts[0], parts[1]

        # 跳过表头
        if name in ("因子名称", "Factor Name", "名称"):
            continue

        # 清理公式两侧的反引号
        formula = formula.strip()
        if formula.startswith("`") and formula.endswith("`"):
            formula = formula[1:-1].strip()

        # 反转义 '\|' → '|'
        formula = formula.replace(r"\|", "|")

        if name and formula:
            rows.append(f"{name}: {formula}")

    # 如果没解析出任何行，就退回原文，避免写入空历史
    return "\n".join(rows) if rows else md_text.strip()

##### 增加历史记忆

In [6]:
# ====================== 历史记忆：简单版 ======================
HISTORY_FILE = "mydata/output/llm_output/factor_gpt_history_v7.txt"

# def append_factor_history(text: str, path: str = HISTORY_FILE) -> None:
#     os.makedirs(os.path.dirname(path), exist_ok=True)
#     with open(path, "a", encoding="utf-8") as f:
#         f.write("\n\n" + "="*80 + f"\n# ROUND @ {datetime.now():%Y-%m-%d %H:%M:%S}\n\n")
#         f.write(text.strip())
def append_factor_history(text: str, path: str = HISTORY_FILE) -> None:
    """
    将本轮 LLM 生成的因子表格写入历史文件。
    写入前先用 compress_factor_table 做瘦身，只保留“因子名称: 公式”。
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)

    # 🌟 核心：先瘦身，再写入
    slim_text = compress_factor_table(text)

    with open(path, "a", encoding="utf-8") as f:
        f.write("\n\n" + "="*80 + f"\n# ROUND @ {datetime.now():%Y-%m-%d %H:%M:%S}\n\n")
        f.write(slim_text.strip())


def load_factor_history(max_rounds: int = 30, path: str = HISTORY_FILE) -> str:
    """读取最近 max_rounds 段历史（按 '# ROUND' 分割）。不存在则返回空串。"""
    if not os.path.exists(path):
        return ""
    with open(path, "r", encoding="utf-8") as f:
        txt = f.read()
    blocks = [b.strip() for b in txt.split("\n# ROUND ") if b.strip()]
    if not blocks:
        return ""
    return "\n\n".join("# ROUND " + b for b in blocks[-max_rounds:])

def build_template1_with_memory(history_text: str, base_prompt: str) -> str:
    """把历史放最前，附上 3 条硬约束，再拼原有的 template1 内容。"""
    guard = (
        "【历史回避清单 / Do-Not-Repeat】\n"
        "下方是过往已生成的因子表（这里只展示名称与计算方式）。本轮必须产出**全新**的因子：\n"
        "1) 不得复用下方任何因子名称；2) 计算公式需有**实质差异**（算子组合或窗口段不同）；\n"
        "3) 若发现重复，请即时替换，仍需输出完整数量。\n"
    )
    if history_text:
        return guard + "\n\n" + history_text + "\n\n" + base_prompt.strip()
    return base_prompt.strip()
# =============================================================

#### 加载数据（lazyfactordb）

In [7]:
import pandas as pd
import numpy as np
import os
import glob
import gc

class LazyFactorDB:
    """
    懒加载
    """
    def __init__(self, data_folders, snapshot_days, max_age=150):
        self.file_map = {}  
        self.cache = {}
        self.snapshot_days = pd.to_datetime(snapshot_days).sort_values()
        self.max_age = max_age
        
        # 【核心修改 1】：兼容处理。如果是传了单个字符串，自动把它包装成列表
        if isinstance(data_folders, str):
            data_folders = [data_folders]
            
        # 【核心修改 2】：遍历所有的主文件夹路径
        for folder in data_folders:
            if not os.path.exists(folder):
                print(f"⚠️ Warning: 路径不存在，已跳过 -> {folder}")
                continue
                
            # 在当前循环的文件夹下进行扫描
            csv_files = glob.glob(os.path.join(folder, "*", "*.csv"))
            for file_path in csv_files:
                metric_name = os.path.splitext(os.path.basename(file_path))[0]
                
                # 提示：如果不同文件夹里出现了同名的 CSV，后扫描到的会覆盖前面的
                if metric_name in self.file_map:
                    print(f"⚠️ 注意: 发现同名科目被覆盖 -> {metric_name} (新路径: {file_path})")
                    
                self.file_map[metric_name] = file_path
                
        print(f"✅ [DB Init] 宽表引擎初始化: 扫描了 {len(data_folders)} 个主目录，共索引 {len(self.file_map)} 个科目 (采样点: {len(self.snapshot_days)} | 过期阈值: {max_age}天)")

    def __getitem__(self, key):
        if key in self.cache: 
            return self.cache[key]
            
        if key not in self.file_map: 
            raise KeyError(f"Factor '{key}' not found in DB.")
        
        file_path = self.file_map[key]
        
        # 1. 极速读取宽表 (全当字符串读，防止混合类型警告)
        df = pd.read_csv(file_path, index_col=0, dtype=str)
        
        # 2. [逻辑变更2]: 矩阵转置 (因为你的原数据是行=股票，列=日期)
        df = df.T
        
        # 3. 数据清洗与格式化
        # 转置后 index 是日期字符串，强转为 Datetime
        df.index = pd.to_datetime(df.index, errors='coerce')
        # 剔除无法识别的烂日期
        df = df[df.index.notna()]
        # 强转为数值型，无法转换的脏字符(如 '--')会变成 NaN
        df = df.apply(pd.to_numeric, errors='coerce')
        
        # 确保时间轴是有序的，去除可能重复的列名/日期
        df = df.sort_index()
        df = df[~df.index.duplicated(keep='last')]
        
        # 4. [黑科技]: 极简的时序对齐与过期剔除
        # reindex 可以完美实现：按 self.snapshot_days 抽取数据，
        # 如果当天没数据，往前 ffill，但最多往前找 max_age 天 (超过则填 NaN)
        final_dense = df.reindex(
            self.snapshot_days,
            method='ffill',
            tolerance=pd.Timedelta(days=self.max_age)
        )
        
        # 5. 降低内存占用并存入缓存
        self.cache[key] = final_dense.astype('float32')
        return self.cache[key]

    def keys(self): 
        return self.file_map.keys()

    def clear_cache(self):
        self.cache.clear()
        gc.collect()
        print("🧹 [Cache] 内存缓存已彻底清理。")

In [8]:
import pandas as pd
import os

# ================= 配置区域 =================
# 👇 请修改这里：指向你任意一个已经合成好的“宽表CSV”的具体文件路径
# 注意：这里直接指向具体的 .csv 文件，而不是文件夹
TEST_FILE_PATH = "/storage/server/145server/ly/luodan/财务数据/指标_v0/LC_BalanceSheetAll/BS_BILLACCRECEIVABLE.csv"
# "mydata/fundamental_data/LC_BalanceSheetAll/BS_BILLACCRECEIVABLE.csv"
# ===========================================

def quick_validate_wide_table(file_path):
    print(f"🔍 开始验证宽表文件: {os.path.basename(file_path)} ...")
    
    if not os.path.exists(file_path):
        print("❌ 错误：文件不存在，请检查路径！")
        return

    try:
        # 1. 测试读取与转置
        print("⏳ [1/3] 正在读取数据并执行矩阵转置...")
        df = pd.read_csv(file_path, index_col=0, dtype=str)
        df = df.T  # 核心动作：把 列(日期) 转成 行
        
        # 2. 测试时间轴解析
        print("⏳ [2/3] 正在解析时间轴 (Index)...")
        original_cols = len(df.index)
        # 尝试把转置后的 index（原来的列名）转为时间戳
        df.index = pd.to_datetime(df.index, errors='coerce')
        valid_dates = df.index.notna().sum()
        
        if valid_dates == 0:
            print("❌ 错误：时间轴完全无法解析！请检查原宽表的表头是否为日期格式。")
            return
        else:
            print(f"  ✅ 时间解析成功: 原本有 {original_cols} 列，成功识别出 {valid_dates} 个有效日期。")
            # 剔除无效日期（比如可能存在的 Unnamed 列）
            df = df[df.index.notna()]
            print(f"  📅 数据区间: {df.index.min().date()} 至 {df.index.max().date()}")

        # 3. 测试数值转换
        print("⏳ [3/3] 正在执行数值净化 (清理 '--' 等脏字符)...")
        df = df.apply(pd.to_numeric, errors='coerce')
        
        # 统计清洗出了多少个 NaN
        total_cells = df.size
        nan_cells = df.isna().sum().sum()
        fill_rate = 1 - (nan_cells / total_cells)
        
        print(f"  ✅ 数值转换完成！")
        print(f"  📐 最终矩阵形状: {df.shape[0]} 个交易日(行) x {df.shape[1]} 只股票(列)")
        print(f"  💧 矩阵非空填充率: {fill_rate:.2%}")
        
        print("\n🎉 验证完美通过！这张宽表的结构完全健康，可以丝滑接入新的 LazyFactorDB 引擎。")
        print("\n📊 预览转置后的标准数据片段 (前5个日期, 前5只股票):")
        print("-" * 60)
        # 打印左上角 5x5 的矩阵切片
        print(df.iloc[:5, :5])
        print("-" * 60)
        
    except Exception as e:
        print(f"\n❌ 验证过程中出现异常: {e}")

# 执行验证
quick_validate_wide_table(TEST_FILE_PATH)

🔍 开始验证宽表文件: BS_BILLACCRECEIVABLE.csv ...
⏳ [1/3] 正在读取数据并执行矩阵转置...
⏳ [2/3] 正在解析时间轴 (Index)...
  ✅ 时间解析成功: 原本有 2430 列，成功识别出 2430 个有效日期。
  📅 数据区间: 2016-01-04 至 2025-12-31
⏳ [3/3] 正在执行数值净化 (清理 '--' 等脏字符)...
  ✅ 数值转换完成！
  📐 最终矩阵形状: 2430 个交易日(行) x 4949 只股票(列)
  💧 矩阵非空填充率: 74.50%

🎉 验证完美通过！这张宽表的结构完全健康，可以丝滑接入新的 LazyFactorDB 引擎。

📊 预览转置后的标准数据片段 (前5个日期, 前5只股票):
------------------------------------------------------------
code                   1             2           4           5            6
2016-01-04  7.120000e+09  1.508193e+09  1748471.19  2495914.47  16437059.97
2016-01-05  7.120000e+09  1.508193e+09  1748471.19  2495914.47  16437059.97
2016-01-06  7.120000e+09  1.508193e+09  1748471.19  2495914.47  16437059.97
2016-01-07  7.120000e+09  1.508193e+09  1748471.19  2495914.47  16437059.97
2016-01-08  7.120000e+09  1.508193e+09  1748471.19  2495914.47  16437059.97
------------------------------------------------------------


In [9]:
import inspect
import re
from tqdm import tqdm
import pandas as pd

class FactorExecutor:
    """
    从大模型的代码中进行
    """
    def __init__(self, db):
        self.db = db
        
    def run(self, calculators, code_text=None):
        """
        【回归 V1 模式】智能执行器：
        1. 分析依赖
        2. 按需提取 Wide DataFrame
        3. 打包成 Context 字典传给算子
        """
        print(f"⚙️ [Executor] 启动 V1 模式计算引擎 (Context Dict)...")
        
        needed_fields = set()
        print("🕵️ 正在分析算子依赖字段...")

        # --- 策略 A: 优先分析原始代码文本 ---
        if code_text:
            matches = re.findall(r"['\"]([a-zA-Z0-9_]+)['\"]", code_text)
            for m in matches:
                # 忽略大小写匹配数据库键
                if m in self.db.file_map or m.lower() in self.db.file_map:
                    needed_fields.add(m)
                    
        # --- 策略 B: 反射 (兜底) ---
        else:
            for name, func in calculators.items():
                try:
                    src = inspect.getsource(func)
                    matches = re.findall(r"['\"]([a-zA-Z0-9_]+)['\"]", src)
                    for m in matches:
                        if m in self.db.file_map or m.lower() in self.db.file_map:
                            needed_fields.add(m)
                except Exception:
                    pass 
        
        # 总是加载 is_listed (用于过滤)
        if 'is_listed' in self.db.file_map:
            needed_fields.add('is_listed')
            
        print(f"   -> 🎯 命中 {len(needed_fields)} 个基础因子: {list(needed_fields)[:5]}...")
        
        if not needed_fields:
            print("❌ 警告：未识别到任何有效依赖，计算可能失败。")
            return pd.DataFrame()

        # --- 第二步：构建 Context 字典 (关键修改：不合并，只打包) ---
        print("💧 正在准备 Context 数据 (不合并，保持独立矩阵)...")
        context_dict = {}
        
        for field in tqdm(needed_fields, desc="Loading Matrices"):
            try:
                # 直接获取宽表 (T x N)，不 stack，不合并
                df_wide = self.db[field]
                context_dict[field] = df_wide
            except Exception as e:
                print(f"❌ 加载 {field} 失败: {e}")

        if not context_dict:
            return pd.DataFrame()
            
        print(f"✅ Context 字典构建完成，包含 {len(context_dict)} 个矩阵。")

        # --- 第三步：准备过滤掩码 ---
        mask_df = None
        if 'is_listed' in context_dict:
             mask_df = context_dict['is_listed']

        # --- 第四步：执行计算 ---
        if 'apply_calculators_vectorized' in globals():
            # 注意：这里传入的是 context_dict，不再是 full_df
            result = apply_calculators_vectorized(context_dict, calculators, is_listed_mask=mask_df)
            return result
        else:
            raise NameError("apply_calculators_vectorized 未定义！")

In [10]:
# 把你所有的财务、量价、另类数据文件夹都放进一个列表里
my_data_paths = [
    "/storage/server/145server/ly/luodan/财务数据/指标_v1",           # 路径1：财务
    # "/storage/server/145server/ly/luodan/财务数据/指标_v0/LC_CashFlowStatementAll", 
    # "/storage/server/145server/ly/luodan/财务数据/指标_v0/LC_IncomeStatementAll",   
    # "/storage/server/145server/ly/luodan/量价指标",           # 路径2：量价
    # "/storage/server/145server/ly/luodan/分析师预期指标"      # 路径3：预期
]

# 传入列表
factor_db = LazyFactorDB(
    data_folders=my_data_paths, 
    snapshot_days=GLOBAL_TRADING_DAYS
)

✅ [DB Init] 宽表引擎初始化: 扫描了 1 个主目录，共索引 544 个科目 (采样点: 30 | 过期阈值: 150天)


#### 大模型输出记录

In [11]:
# -------------------- 文件保存逻辑 --------------------
def save_output_to_file(output: str, file_prefix: str, save_directory: str = 'mydata/output/llm_output/llm_output_v7') -> str:
    os.makedirs(save_directory, exist_ok=True)
    filename = os.path.join(save_directory, f"{file_prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(output)
    print(f"文件已保存: {filename}")
    return filename


#### 构建LLM链

In [12]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [13]:
def run_three_stages_with_memory(client1, client2, history_text,all_fields, extra_instruction=""):
    # Stage 1: Design
    messages_stage1 = [
        {"role": "system", "content": STAGE1_SYSTEM_PROMPT_TEMPLATE},
        {"role": "user", "content": build_stage1_user_content(history_text,all_fields,  extra_instruction)}
    ]
    
    print(">>> Calling Stage 1 (Factor Design)...")
    resp1 = client1.invoke(to_lc_messages(messages_stage1))
    text_1 = getattr(resp1, "content", str(resp1))
    
    save_output_to_file(text_1, file_prefix="model_chain_1_output")
    append_factor_history(text_1)

    # Stage 2: Code Gen
    messages_stage2 = [
        {"role": "system", "content": STAGE2_SYSTEM_PROMPT},
        {"role": "user", "content": build_stage2_user_content(text_1)}
    ]
    
    print(">>> Calling Stage 2 (Code Generation)...")
    resp2 = client2.invoke(to_lc_messages(messages_stage2))
    text_2 = getattr(resp2, "content", str(resp2))
    
    save_output_to_file(text_2, file_prefix="model_chain_2_output")

    # Stage 3: Code Check
    messages_stage3 = [
        {"role": "system", "content": STAGE3_SYSTEM_PROMPT},
        {"role": "user", "content": build_stage3_user_content(text_2,all_fields)}
    ]
    
    print(">>> Calling Stage 3 (Code Correction)...")
    resp3 = client2.invoke(to_lc_messages(messages_stage3))
    text_3 = getattr(resp3, "content", str(resp3))
    
    save_output_to_file(text_3, file_prefix="model_chain_3_output")

    return text_3, text_1


#### get system

In [14]:
# # 再次确认一下，确保 val_ 彻底被排除
# def get_system_prompt_context(db):
#     """
#     【纯财务版】仅暴露三大表 (BS, IS, CFS) 和 元数据。
#     彻底移除估值 (val_) 和行情相关字段。
#     """
#     # 👇👇👇 你漏掉了这一行！这是获取所有文件名的关键 👇👇👇
#     fields = sorted(list(db.keys())) 
#     # 👆👆👆 必须先定义 fields，下面才能用 👆👆👆

#     categories = {
#         "0. 基础元数据 (meta_*)": [f for f in fields if f.startswith('meta_')],
#         "1. 资产负债表 (bs_*)": [f for f in fields if f.startswith('bs_')],
#         "2. 利润表 (is_*)": [f for f in fields if f.startswith('is_')],
#         "3. 现金流量表 (cfs_*)": [f for f in fields if f.startswith('cfs_')],
#     }
    
#     context_str = "【可用数据字段清单 (仅含财务报表)】\n"
#     for cat, items in categories.items():
#         if items:
#             item_str = ", ".join(items)
#             context_str += f"### {cat}:\n[{item_str}]\n\n"
            
#     return context_str

#### 常见算子库

In [15]:
import pandas as pd
import numpy as np
import os
import ast
import traceback

# =========================================================================
# 1. 全局算子定义 (扁平化结构，解决 NameError)
# =========================================================================
EPS = 1e-10
DAYS_PER_QUARTER = 60

# def _ensure_df(x): return x.to_frame() if isinstance(x, pd.Series) else x
# def _check_window(d): return max(1, int(d * DAYS_PER_QUARTER))

# 🔥 核心修改：不再需要 DAYS_PER_QUARTER，所有窗口 d 直接代表“期数”
def _ensure_df(x): return x.to_frame() if isinstance(x, pd.Series) else x
def _check_window(d): return max(1, int(d)) # d=1 就是回溯1个切片

# --- 基础截面 ---
def CS_Rank(x): return _ensure_df(x).rank(axis=1, pct=True)

# --- 高级截面 (行业中性) ---
def CS_Indus_Rank(x, indus_matrix):
    x = _ensure_df(x)
    if indus_matrix is None: return CS_Rank(x)
    try:
        if isinstance(indus_matrix, pd.Series): indus_matrix = indus_matrix.to_frame()
        df_long = pd.concat([x.stack(), indus_matrix.stack()], axis=1, keys=['val', 'ind'], join='inner')
        ranked = df_long.groupby([df_long.index.get_level_values(0), 'ind'])['val'].rank(pct=True)
        return ranked.unstack()
    except: return CS_Rank(x)

# 建议添加到算子库
def TTM(x):
    """滚动12个月求和 (假设数据为单季度值)"""
    return x.rolling(_check_window(3)).sum()

# --- 元素/数学 ---
# def YOY(x): prev = x.shift(250); return (x - prev) / (prev.abs() + EPS)
def YOY(x): 
    """
    同比 (Year-Over-Year):
    在 4.30/8.30/10.30 的体系下，每年3个点。
    因此去年的同一期，是往前推 3 个单位。
    """
    prev = x.shift(3) 
    return (x - prev) / (prev.abs() + EPS)

# def QOQ(x): prev = x.shift(60); return (x - prev) / (prev.abs() + EPS)
def QOQ(x): 
    """
    环比 (Quarter-Over-Quarter):
    直接与上一期（shift 1）比较。
    注意：在4.30时，对比的是去年10.30的三季报数据（逻辑上虽有跳跃，但这是数据流的真实情况）
    """
    prev = x.shift(1)
    return (x - prev) / (prev.abs() + EPS)
# def Delay(x, d): return x.shift(_check_window(d))
def Delay(x, d): 
    """季频滞后: d=1 代表取上一期财报 (如当前是中报，Delay(1)就是一季报)"""
    return x.shift(int(d))
    
# def Delta(x, d):
#     """时序差分: x_t - x_{t-d}"""
#     return x.diff(int(d))

def Inv(x): return 1.0 / (x + EPS)
def Abs(x): return x.abs()
def Sign(x): return np.sign(x)
def Log(x): return np.log(x.abs() + EPS)
def Exp(x): return np.exp(x.clip(upper=20))
def Sqrt(x): return np.sqrt(x.abs())
def Pow(x, y): return np.sign(x) * (x.abs() ** y)

# --- 关系/复合 ---
def Add(x, y): return x + y
def Sub(x, y): return x - y
def Mul(x, y): return x * y
def Div(x, y): return x / (y + EPS)
def Rank_Add(x, y): return CS_Rank(x) + CS_Rank(y)
def Rank_Sub(x, y): return CS_Rank(x) - CS_Rank(y)
def Rank_Mul(x, y): return CS_Rank(x) * CS_Rank(y)
def Rank_Div(x, y): return CS_Rank(x) / (CS_Rank(y) + EPS)

# --- 时序 ---
def TS_Mean(x, d): return x.rolling(_check_window(d)).mean()
def TS_Std(x, d): return x.rolling(_check_window(d)).std()
def TS_Sum(x, d): return x.rolling(_check_window(d)).sum()
def TS_Corr(x, y, d): return x.rolling(_check_window(d)).corr(y)
# def ts_delay(x, d): return x.shift(int(d)) # 日频专用
def delta(x, d):    return x.diff(_check_window(d))

# --- 核心修复：增加 Delta 算子 ---
def Delta(x, d): 
    """时序差分：x_t - x_{t-d}"""
    return x.diff(_check_window(d))

# Delta = delta
# --- [新增] 极值与位置 (华泰图表9要求) ---
def TS_Min(x, d): return x.rolling(_check_window(d)).min()
def TS_Max(x, d): return x.rolling(_check_window(d)).max()
def TS_Argmin(x, d): return x.rolling(_check_window(d)).apply(np.argmin) # 最小值发生的位置
def TS_Argmax(x, d): return x.rolling(_check_window(d)).apply(np.argmax) # 最大值发生的位置

# --- [新增] 趋势与回归 (东吴Alpha158常用) ---
# 注意：滚动回归计算较慢，如果追求速度可暂时不加，但对挖掘Alpha很重要
def Slope(x, d):
    """计算x在过去d天的线性回归斜率 (简单实现版)"""
    w = _check_window(d)
    # 使用 numpy polyfit 的简化版或 pandas 的 cov/var 实现
    # Slope = Cov(x, t) / Var(t)
    # 这里为了性能，通常简化为时序上的变化率，或者用 rolling_apply
    # 既然是因子挖掘，建议先用简单的 Delta 代替，或者使用以下 pandas 实现：
    return x.diff(w) / w # 简易版斜率，如果要精确回归斜率需用 rolling().apply

# 修正 TTM 逻辑 (重要提醒)
# 如果你的数据是日频(每天都有值)，且进行了向前填充。
# rolling(240).sum() 会把每一天的值都加起来，导致数值巨大错误。
# 建议 TTM 改为如下逻辑 (假设 x 是单季度值，且每季度只变一次):
# 或者，最安全的做法是让 AI 使用 TS_Sum(x, 4) 但明确 x 必须是稀疏数据。
# 鉴于日频数据处理的复杂性，建议暂时保留你现有的 TTM，但需确保输入数据的频率。

# --- 算子打包函数 ---
def matrix_operators():
    return {k: v for k, v in globals().items() if callable(v) and not k.startswith('_')}

##### 旧版算子库j

###### 基本面算子

#### 创建独立模块进行运算

##### 保存出错的因子

In [16]:
#----------------------保存出错因子信息--------------------
from datetime import datetime
import os

def log_factor_error(factor_name, stock_code, error_message, log_path="mydata/output/factor/日频因子v7/factor_error_log.txt"):
    """
    将因子计算错误信息写入日志文件。
    """
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(
            f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
            f"因子: {factor_name:<30} | 股票: {stock_code:<10} | 错误: {error_message}\n"
        )

##### 计算因子值

In [17]:
from tqdm import tqdm
import pandas as pd

# 👇 注意这里的返回值类型注解变成了 dict
def apply_calculators_vectorized(context: dict, calculators: dict, is_listed_mask: pd.DataFrame = None) -> dict:
    """
    【V1 纯宽表模式】
    输入 context: 字典 {'open': df_wide, ...}
    输出: 字典 {'FactorName': df_wide, ...}
    """
    
    # 1. 掩码预处理
    aligned_mask = is_listed_mask 
    if aligned_mask is not None:
        print("🧹 [Zombie Filter] 已启用僵尸数据实时过滤 (Matrix Mode)...")

    print(f"🚀 开始计算 {len(calculators)} 个因子 (Matrix Mode)...")
    
    # 👇 改动点1：容器变成了一个字典，而不是列表
    # 结果容器：直接用字典存储宽表
    calculated_factors = {}

    for name, func in tqdm(calculators.items(), desc="Computing"):
        factor_name = name.replace("calculate_", "")
        try:
            # === A. 执行计算 ===
            res = func(context)
            
            # === B. 过滤 ===
            if aligned_mask is not None and isinstance(res, pd.DataFrame):
                res = res * aligned_mask

            # === C. 收集结果 ===
            # 👇 改动点2：直接把结果存入字典，而不是 append 到 list 后再 concat
            # 直接存入字典，保持 Wide Format (Date x Code)
            if isinstance(res, (pd.DataFrame, pd.Series, int, float)):
                calculated_factors[factor_name] = res
            else:
                if res is not None:
                    print(f"⚠️ 警告: 因子 {factor_name} 返回类型为 {type(res)}，跳过。")
                
        except Exception as e:
            print(f"❌ 因子 {factor_name} 计算失败: {e}")
            
    # 🔥🔥🔥 改动点3 (最重要)：直接返回字典 🔥🔥🔥
    # 旧版本这里通常是 pd.concat(...) 返回一个巨大的 DataFrame
    return calculated_factors

In [18]:
def load_daily_mask(db, factor_name='is_listed'):
    """
    [季频专用版] 加载掩码
    关键点：
    1. 不做 ffill (前向填充)，因为季频本身就是离散的。
    2. 严格对齐到 GLOBAL_TRADING_DAYS。
    """
    if factor_name not in db.file_map:
        print(f"❌ 错误：找不到文件 {factor_name}，将不启用过滤。")
        return None
        
    print(f"🛡️ [季频] 正在加载掩码: {factor_name}...")
    path = db.file_map[factor_name]
    
    # 1. 读取
    df_long = pd.read_parquet(path)
    
    # 2. Pivot
    cols = df_long.columns
    # 自动寻找列名
    date_col = next((c for c in cols if 'DATE' in c.upper() and 'PUBL' in c.upper()), 'INFOPUBLDATE')
    code_col = next((c for c in cols if 'CODE' in c.upper()), 'SECUCODE')
    val_col = [c for c in cols if c not in [date_col, code_col]][0]
    
    df_wide = df_long.pivot(index=date_col, columns=code_col, values=val_col)
    df_wide.index = pd.to_datetime(df_wide.index)
    
    # 3. 严格对齐 (不做 ffill)
    # 只有在 GLOBAL_TRADING_DAYS (4/30, 8/30...) 有值的才保留
    df_mask = df_wide.reindex(db.trading_days)
    
    # 4. 转布尔值
    df_bool = (df_mask > 0).astype(float) 
    
    print(f"✅ 掩码加载完毕，覆盖率: {df_bool.mean().mean():.2%}")
    return df_bool

#### 分组保存计算好的因子值

In [19]:
# -------------------- 合并指定类型的 chain 输出（支持 model_chain_1/2/3...） --------------------
def combine_model_chain_outputs(
    input_dir: str = 'mydata/output/llm_output/llm_output_v7',
    output_dir: str = 'mydata/output/llm_output/combined_v7',
    chain_prefix: str = "model_chain_3_output_",  # 如 "model_chain_2_output_"
    # chain_prefix:str,
    include_timestamp: bool = True
):
    """
    将 input_dir 中所有以 {chain_prefix}*_.txt 结尾的文件按时间戳排序，
    合并到 output_dir 下对应的 combined_{chain_prefix}.txt 文件中。

    参数：
        input_dir: 原始输出文件所在目录
        output_dir: 合并后的文件保存目录
        chain_prefix: 要合并的文件前缀，如 "model_chain_1_output_"
        include_timestamp: 是否在合并内容中标注时间
    """
    import os
    import glob
    from datetime import datetime

    # 确保输出目录存在
    os.makedirs(output_dir, exist_ok=True)

    # 构建搜索模式：model_chain_1_output_YYYYMMDD_HHMMSS.txt
    pattern = os.path.join(input_dir, f"{chain_prefix}*_*.txt")
    files = glob.glob(pattern)

    if not files:
        print(f"⚠️ 在 {input_dir} 中未找到任何匹配 {chain_prefix}*_.txt 的文件。")
        return

    def extract_time(filepath):
        basename = os.path.basename(filepath)
        try:
            # 提取 time_str: model_chain_1_output_20251110_163759.txt -> 20251110_163759
            time_str = basename.replace(chain_prefix, "").replace(".txt", "")
            date_part, time_part = time_str.split("_")
            dt_str = f"{date_part}_{time_part}"
            return datetime.strptime(dt_str, "%Y%m%d_%H%M%S")
        except Exception as e:
            print(f"无法解析时间戳: {basename}, 错误: {e}")
            return datetime.min

    # 按时间排序
    sorted_files = sorted(files, key=extract_time)

    # 输出文件名基于 chain_prefix 定制
    safe_prefix = chain_prefix.rstrip("_")  # 去掉末尾下划线便于命名
    output_file = os.path.join(output_dir, f"combined_{safe_prefix}.txt")

    print(f"📦 发现 {len(sorted_files)} 个 '{chain_prefix}' 类型文件，开始合并至:\n   {output_file}")

    with open(output_file, 'w', encoding='utf-8') as out_f:
        for file_path in sorted_files:
            basename = os.path.basename(file_path)
            timestamp_str = basename.replace(chain_prefix, "").replace(".txt", "")
            date_part, time_part = timestamp_str.split("_")
            formatted_time = f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:]} {time_part[:2]}:{time_part[2:4]}:{time_part[4:]}"

            try:
                with open(file_path, 'r', encoding='utf-8') as in_f:
                    content = in_f.read().strip()
                    if not content:
                        content = "<空文件>"

                # 写入分隔块
                out_f.write(f"\n{'='*60}\n")
                out_f.write(f"📌 源文件: {basename}\n")
                if include_timestamp:
                    out_f.write(f"🕒 生成时间: {formatted_time}\n")
                out_f.write(f"{'-'*60}\n")
                out_f.write(content)
                out_f.write(f"\n{'='*60}\n\n")

            except Exception as e:
                print(f"❌ 读取文件失败: {file_path}, 错误: {e}")

    print(f"✅ 已完成合并: {len(sorted_files)} 个文件 → {output_file}")

In [20]:
def save_factor_to_directory(df: pd.DataFrame, base_out: str) -> None:
    # 1️⃣ 预处理
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(axis=1, how='all')
    all_null_columns = df.columns[df.isnull().all()].tolist()
    print(f"被删除的全空值列: {all_null_columns}")

    Path(base_out).mkdir(parents=True, exist_ok=True)
    factor_cols = [c for c in df.columns if c not in need_cols]

    def safe_name(name: str) -> str:
        return re.sub(r'[\\/:"*?<>|]+', "_", str(name))

    # 2️⃣ 为“本次函数调用”创建一个时间戳目录
    #    加上微秒，避免循环里在同一秒内多次调用撞名
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    run_dir = Path(base_out) / ts_str
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"本次保存目录: {run_dir}")

    # 3️⃣ 在时间戳目录下：按因子拆子文件夹，再按日期拆 feather
    for fac in factor_cols:
        sub = df[['date', 'code', fac]].copy()

        # 因子目录： base_out / <时间戳> / <因子名(转义后)>
        fac_dir = run_dir / safe_name(fac)
        fac_dir.mkdir(parents=True, exist_ok=True)

        # 按日期分组保存
        for day_str, g in sub.groupby(sub['date'].dt.strftime('%Y-%m-%d'), sort=True):
            out_file = fac_dir / f"{day_str}.feather"
            to_write = g[['code', fac]].sort_values('code', kind='mergesort').reset_index(drop=True)
            to_write.to_feather(out_file)

    print("✅ 本次运行已按【时间戳/因子/日期】结构保存完成（Feather 格式）！")


#### 算子库拼接

In [21]:
# all_fields

In [22]:
# =================================================================
# 1. 定义算子库头文件 (用于发送给远程平台)
# =================================================================
# 【核心原则】本地 matrix_operators 有什么，这里就必须有什么
# =================================================================

OPERATOR_HEADER_STR = r'''
import pandas as pd
import numpy as np

# --- 0. 核心装饰器 ---
# ================= 0. 核心装饰器: 双向数据适配器 (动态提取版) =================
def auto_process(func):
    def wrapper(data, *args, **kwargs):
        try:
            # [模式 A] 本地调用: 输入已经是矩阵字典 -> 直接计算，直接返回矩阵
            if isinstance(data, dict):
                return func(data, *args, **kwargs)
            
            # [模式 B] 远程调用: 输入是长表 DataFrame -> "拆解计算，组装返回"
            if isinstance(data, pd.DataFrame):
                # --- 1. Input Adapter: 长表 -> 矩阵字典 ---
                df_long = data.copy()
                # 记录原始索引，用于最后还原对齐
                original_index = df_long.index
                
                # 确保有 date/code 列用于 Pivot
                if isinstance(df_long.index, pd.MultiIndex):
                    df_long = df_long.reset_index()
                
                # 识别 date 和 code 列名 (忽略大小写进行匹配，但保留原表列名)
                date_col = next((c for c in df_long.columns if 'date' in str(c).lower()), 'date')
                code_col = next((c for c in df_long.columns if 'code' in str(c).lower() or 'asset' in str(c).lower()), 'code')
                
                matrix_dict = {}
                
                if date_col in df_long.columns and code_col in df_long.columns:
                    # 统一转 datetime 方便排序
                    df_long[date_col] = pd.to_datetime(df_long[date_col])
                    
                    # 🚀 核心优化：动态提取目标字段（排除坐标轴后，剩下的全是财务科目）
                    exclude_cols = {date_col, code_col}
                    target_fields = [c for c in df_long.columns if c not in exclude_cols]
                    
                    for field in target_fields:
                        # Pivot: Index=Date, Columns=Code, Values=原始列名
                        matrix = df_long.pivot(index=date_col, columns=code_col, values=field)
                        matrix = matrix.sort_index()
                        
                        # 存入字典（直接使用原始列名作为 Key）
                        matrix_dict[field] = matrix

                # --- 2. Core Execution: 执行矩阵运算 ---
                if not matrix_dict:
                    # 没提取到数据，尝试直接传原数据(死马当活马医)
                    print("⚠️ [AutoProcess Warning] 未能生成宽表字典，降级透传原表。")
                    return func(data, *args, **kwargs)
                
                # 执行你的因子计算逻辑
                result_matrix = func(matrix_dict, *args, **kwargs)
                
                # --- 3. Output Adapter: 矩阵 -> 长表 Series (闭环还原) ---
                # 平台期望得到一个与输入 data 索引一一对应的 Series
                if isinstance(result_matrix, pd.DataFrame):
                    # 3.1 宽变长 (Stack): 变成 (Date, Code) 的 Series
                    # stack() 自动忽略 NaN
                    series_long = result_matrix.stack()
                    
                    # 3.2 构造目标索引 (Target Index)
                    target_df = df_long[[date_col, code_col]].copy()
                    
                    series_name = '_factor_temp_'
                    series_long.name = series_name
                    
                    # 变成 DataFrame: Index=(Date, Code), Value=Factor
                    factor_long_df = series_long.reset_index()
                    factor_long_df.columns = [date_col, code_col, series_name]
                    
                    # 3.3 Merge 回原始顺序 (使用 left join 保证行数和顺序严格一致)
                    merged = pd.merge(target_df, factor_long_df, on=[date_col, code_col], how='left')
                    
                    # 3.4 提取 Series 并恢复原始索引
                    result_series = merged[series_name]
                    result_series.index = original_index 
                    
                    return result_series
                
                # 如果返回的不是 DF (比如是常数或已有 Series)，直接返回
                return result_matrix

            # 其他情况直接透传
            return func(data, *args, **kwargs)

        except Exception as e:
            print(f"❌ [AutoProcess Error] {str(e)}")
            raise e
            
    return wrapper

# --- 1. 全局配置 ---
EPS = 1e-10
DAYS_PER_QUARTER = 60

def _ensure_df(x): return x.to_frame() if isinstance(x, pd.Series) else x
def _check_window(d): return max(1, int(d)) # d=1 就是回溯1个切片

# --- 2. 截面算子 ---
def CS_Rank(x): return _ensure_df(x).rank(axis=1, pct=True)

def CS_Indus_Rank(x, indus_matrix):
    x = _ensure_df(x)
    if indus_matrix is None: return CS_Rank(x)
    try:
        if isinstance(indus_matrix, pd.Series): indus_matrix = indus_matrix.to_frame()
        df_long = pd.concat([x.stack(), indus_matrix.stack()], axis=1, keys=['val', 'ind'], join='inner')
        ranked = df_long.groupby([df_long.index.get_level_values(0), 'ind'])['val'].rank(pct=True)
        return ranked.unstack()
    except: return CS_Rank(x)

# --- 3. 基础变换 ---
def TTM(x): 
    """[新增] 滚动12个月求和"""
    return x.rolling(_check_window(3)).sum()

def YOY(x): prev = x.shift(3); return (x - prev) / (prev.abs() + EPS)
def QOQ(x): prev = x.shift(1); return (x - prev) / (prev.abs() + EPS)
def Delay(x, d): return x.shift(_check_window(d))
def Delta(x, d): return x.diff(_check_window(d))

def Inv(x): return 1.0 / (x + EPS)
def Abs(x): return x.abs()
def Sign(x): return np.sign(x)
def Log(x): return np.log(x.abs() + EPS)
def Exp(x): return np.exp(x.clip(upper=20))
def Sqrt(x): return np.sqrt(x.abs())
def Pow(x, y): return np.sign(x) * (x.abs() ** y)

# --- 4. 复合运算 ---
def Add(x, y): return x + y
def Sub(x, y): return x - y
def Mul(x, y): return x * y
def Div(x, y): return x / (y + EPS)
def Rank_Add(x, y): return CS_Rank(x) + CS_Rank(y)
def Rank_Sub(x, y): return CS_Rank(x) - CS_Rank(y)
def Rank_Mul(x, y): return CS_Rank(x) * CS_Rank(y)
def Rank_Div(x, y): return CS_Rank(x) / (CS_Rank(y) + EPS)

# --- 5. 时序统计 (补全华泰研报需求) ---
def TS_Mean(x, d): return x.rolling(int(d)).mean()
def TS_Std(x, d): return x.rolling(int(d)).std()
def TS_Sum(x, d): return x.rolling(int(d)).sum()
def TS_Corr(x, y, d): return x.rolling(int(d)).corr(y)

# [新增] 极值类算子
def TS_Min(x, d): return x.rolling(int(d)).min()
def TS_Max(x, d): return x.rolling(int(d)).max()
def TS_Argmin(x, d): return x.rolling(_check_window(d)).apply(np.argmin, raw=True)
def TS_Argmax(x, d): return x.rolling(_check_window(d)).apply(np.argmax, raw=True)

def Slope(x, d):
    """计算x在过去d天的线性回归斜率 (简单实现版)"""
    w = _check_window(d)
    # 使用 numpy polyfit 的简化版或 pandas 的 cov/var 实现
    # Slope = Cov(x, t) / Var(t)
    # 这里为了性能，通常简化为时序上的变化率，或者用 rolling_apply
    # 既然是因子挖掘，建议先用简单的 Delta 代替，或者使用以下 pandas 实现：
    return x.diff(w) / w # 简易版斜率，如果要精确回归斜率需用 rolling().apply

# 兼容性别名
delta = Delta
delay = Delay
'''

In [23]:
import re

def pack_code_for_remote(header_str, strategy_code):
    """
    将算子定义与策略代码合并，并自动给计算函数加上 @auto_process 装饰器
    """
    # 1. 自动注入装饰器
    # 找到所有 def calculate_xxx(context): 替换为 @auto_process \n def calculate...
    strategy_code_fixed = re.sub(
        r"(def\s+calculate_)", 
        r"@auto_process\n\1", 
        strategy_code
    )
    
    # 2. 拼接
    final_script = f"{header_str}\n\n# ================= STRATEGY =================\n{strategy_code_fixed}"
    return final_script

In [24]:
OPERATOR_DOCS = """
# =============================================================================
# 【研报标准算子库 (Standard Operator List)】
# 变量说明：X, Y 为矩阵 (DataFrame)；d 为时间窗口 (季度数)。
# =============================================================================

# 1. 元素算子 (Element-wise)
def YOY(X): '''同比'''
def QOQ(X): '''环比'''
def Delay(X, d): '''滞后 d 个季度'''
def Inv(X): '''倒数 1/X'''
def Log(X): '''对数'''
def Abs(X): '''绝对值'''
def Sqrt(X): '''平方根'''

# 2. 截面算子 (Cross-Section)
def CS_Rank(X): '''全市场截面排序 (0~1)'''

# 3. 复合关系算子 (Composite Relational)
# 核心逻辑：先分别做 Rank，再进行四则运算。
def Rank_Add(X, Y): '''rank(X) + rank(Y)'''
def Rank_Sub(X, Y): '''rank(X) - rank(Y)'''
def Rank_Mul(X, Y): '''rank(X) * rank(Y)'''
def Rank_Div(X, Y): '''rank(X) / rank(Y)'''
# 基础运算
def Add(X, Y): '''X + Y'''
def Sub(X, Y): '''X - Y'''
def Mul(X, Y): '''X * Y'''
def Div(X, Y): '''X / Y'''

# 4. 时序算子 (Time-Series)
# 注意：d 代表“季度数”。底层会自动转化为 d*60 天。
def TS_Mean(X, d): '''过去 d 个季度的均值'''
def TS_Std(X, d):  '''过去 d 个季度的标准差'''
def TS_Sum(X, d):  '''过去 d 个季度的总和'''
def TS_Corr(X, Y, d): '''过去 d 个季度的相关系数'''
"""

In [25]:
# final_code = combine_operator_lib_with_strategy(operator_lib_header1, code)

In [26]:
# print(final_code)

In [27]:
import re

def combine_operator_lib_with_strategy(operator_lib_header1, strategy_code):
    """
    拼接算子库与策略代码，并自动注入装饰器。
    Args:
        operator_lib_header1: 包含算子定义和 @auto_process 装饰器的完整字符串
        strategy_code: LLM 生成的策略代码
    """
    
    # 使用正则全局替换：给所有 calculate_ 函数戴上帽子
    strategy_code_fixed = re.sub(
        r"(def\s+calculate_)", 
        r"@auto_process\n\1", 
        strategy_code
    )

    # 最终合并
    final_script = f"{operator_lib_header1}\n\n# ================= STRATEGY =================\n{strategy_code_fixed}"
    
    return final_script

#### 平台组结果轮询与保存

In [28]:
import os
import json
import time
import uuid
import requests
import pandas as pd
import warnings
from datetime import datetime

# ==========================================
# 辅助函数: 智能轮询与保存
# ==========================================
def wait_and_save_remote_result(base_url, job_id, save_path, max_retries=60*15, sleep_time=3):
    """
    轮询远程任务状态，成功后保存结果为 JSON。
    文件名建议包含时间戳，例如: 20251217_1430_remote_job_xxx.json
    """
    target_url = f"{base_url}/jobs/{job_id}"
    print(f"📡 [远程同步] 开始监控任务: {job_id}")
    
    # 确保保存目录存在
    save_dir = os.path.dirname(save_path)
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)
    
    for i in range(max_retries):
        try:
            res = requests.get(target_url)
            if res.status_code == 200:
                res_json = res.json()
                
                # 获取关键字段
                current_status = res_json.get('status')
                results = res_json.get('results')

                # --- 情况 1: 正在运行 ---
                if current_status == 'running':
                    if i % 10 == 0: # 减少刷屏频率
                        print(f"⏳ 平台计算中... (状态: {current_status}, 已耗时: {i*sleep_time}s)")
                
                # --- 情况 2: 成功 (状态完成 或 结果非空) ---
                elif results is not None or current_status in ['success', 'completed', 'finished']:
                    print(f"✅ 平台计算完成! (状态: {current_status})")
                    
                    # 保存完整 JSON (Metadata + Data)
                    with open(save_path, 'w', encoding='utf-8') as f:
                        json.dump(res_json, f, ensure_ascii=False, indent=4)
                    print(f"💾 远程结果已归档: {os.path.basename(save_path)}")
                    return res_json
                
                # --- 情况 3: 明确失败 ---
                elif current_status in ['failed', 'error']:
                    error_msg = res_json.get('payload', {}).get('message', 'Unknown error')
                    print(f"❌ 平台计算失败: {error_msg}")
                    return None
                
                else:
                    print(f"⚠️ 未知状态: {current_status}，继续等待...")

            else:
                print(f"⚠️ 轮询请求异常: {res.status_code}")
                
        except Exception as e:
            print(f"⚠️ 轮询网络错误: {e}")

        # 等待后重试
        time.sleep(sleep_time)

    print("❌ [远程同步] 等待超时，未获取到结果。")
    return None

In [29]:
# import os
# import json
# import time
# import uuid
# import requests
# import pandas as pd
# import warnings
# from datetime import datetime

# def wait_and_save_remote_result(
#     base_url, 
#     job_id, 
#     save_path, 
#     max_wait_hours=2,     # ← 关键：改为“最多等多久”
#     sleep_time=3,
# ):
#     """
#     轮询远程任务，最长等待 max_wait_hours 小时。
#     例如：max_wait_hours=24 → 最多重试 24*3600/3 = 28800 次
#     """
#     max_retries = int(max_wait_hours * 3600 / sleep_time)  # 自动换算
#     print(f"⏳ 最大等待时间: {max_wait_hours} 小时 ({max_retries} 次轮询, 每 {sleep_time}s 一次)")

#     target_url = f"{base_url}/jobs/{job_id}"
#     print(f"📡 [远程同步] 开始监控任务: {job_id} (最长等待 {max_wait_hours} 小时)")

#     # 确保目录存在
#     save_dir = os.path.dirname(save_path)
#     if save_dir and not os.path.exists(save_dir):
#         os.makedirs(save_dir, exist_ok=True)

#     for i in range(max_retries):
#         try:
#             res = requests.get(target_url, timeout=10)
#             if res.status_code == 200:
#                 res_json = res.json()
#                 status = res_json.get('status')
#                 results = res_json.get('results')

#                 # ✅ 成功
#                 if results is not None or status in {'success', 'completed', 'finished'}:
#                     print(f"✅ 完成! (状态: {status})")
#                     with open(save_path, 'w', encoding='utf-8') as f:
#                         json.dump(res_json, f, ensure_ascii=False, indent=4)
#                     print(f"💾 已保存: {os.path.basename(save_path)}")
#                     return res_json

#                 # ❌ 失败
#                 elif status in {'failed', 'error'}:
#                     msg = res_json.get('payload', {}).get('message', 'Unknown error')
#                     print(f"❌ 失败: {msg}")
#                     return None

#                 # ⏳ 运行中：降低日志频率
#                 elif status == 'running':
#                     if i % 20 == 0:  # 每 20 次（约 1 分钟）打一次 log
#                         elapsed_min = (i * sleep_time) // 60
#                         print(f"⏳ 运行中... (已等待 {elapsed_min} 分钟)")

#                 else:
#                     if i % 50 == 0:
#                         print(f"⚠️ 未知状态 '{status}'，继续等待...")

#             else:
#                 if i % 100 == 0:
#                     print(f"⚠️ HTTP {res.status_code}，重试中...")

#         except Exception as e:
#             if i % 100 == 0:  # 避免刷屏
#                 print(f"⚠️ 异常: {e}，继续重试...")

#         time.sleep(sleep_time)

#     # 🔚 超时退出
#     total_wait_min = max_retries * sleep_time // 60
#     print(f"❌ [超时] 轮询 {total_wait_min} 分钟 ({max_retries} 次) 后仍未完成，退出。")
#     return None

#### 切割保存(关联job_id)

In [30]:
# import pandas as pd
# import re
# import os

# def index_and_save_job_details(job_id, llm1_text, llm3_text, save_path="mydata/output/job_id_output/job_id_output_v7.csv"):
#     """
#     【升级版】解析并保存：因子名、原始公式、逻辑解释、实现代码
#     """
    
#     # 1. 解析 LLM1 (Markdown 表格)
#     logic_list = []
#     lines = llm1_text.split('\n')
#     for line in lines:
#         # 过滤有效行
#         if '|' in line and '因子名称' not in line and '---' not in line:
#             parts = [p.strip() for p in line.split('|')]
#             # Markdown 表格通常分割后 index 为: [空, 名字, 公式, 解释, 空]
#             if len(parts) >= 4:
#                 raw_name = parts[1].replace('**', '').replace('`', '').strip()
                
#                 # 🟢 新增：提取公式列 (去除代码反引号 `)
#                 raw_formula = parts[2].replace('`', '').strip()
                
#                 logic_desc = parts[3].strip()
                
#                 logic_list.append({
#                     'Factor_Name': raw_name, 
#                     'Formula': raw_formula,  # <--- 这里保留了公式
#                     'Logic': logic_desc
#                 })
#     df_logic = pd.DataFrame(logic_list)

#     # 2. 解析 LLM3 (Python 代码)
#     code_list = []
#     chunks = llm3_text.split('def ')
#     for chunk in chunks:
#         if not chunk.strip(): continue
#         first_line = chunk.split('\n')[0]
#         match = re.search(r'calculate_(\w+)\(df\):', first_line)
#         if match:
#             factor_name = match.group(1)
#             full_code = "def " + chunk.strip()
#             code_list.append({'Factor_Name': factor_name, 'Code': full_code})
#     df_code = pd.DataFrame(code_list)

#     # 3. 对齐与合并
#     if df_logic.empty or df_code.empty:
#         print(f"⚠️ Job {job_id} 解析为空，请检查输入文本。")
#         return

#     merged_df = pd.merge(df_logic, df_code, on='Factor_Name', how='inner')
    
#     # 4. 生成 ID
#     merged_df = merged_df.sort_values('Factor_Name').reset_index(drop=True)
# #     merged_df['Unique_ID'] = [f"{job_id}_{i+1:02d}" for i in range(len(merged_df))]
#     merged_df['Job_ID'] = job_id

#     # 5. 保存 (列顺序优化)
#     # 现在的表格将非常完美：ID -> 名字 -> 公式 -> 解释 -> 代码
# #     save_cols = ['Unique_ID', 'Job_ID', 'Factor_Name', 'Formula', 'Logic', 'Code']
#     save_cols = ['Job_ID', 'Factor_Name', 'Formula', 'Logic', 'Code']
    
#     os.makedirs(os.path.dirname(save_path), exist_ok=True)
#     header = not os.path.exists(save_path)
#     merged_df[save_cols].to_csv(save_path, mode='a', header=header, index=False, encoding='utf-8-sig')
    
#     print(f"✅ [索引更新] Job {job_id} 已录入 {len(merged_df)} 个因子 (含公式列)。")

In [31]:
import pandas as pd
import re
import os

def index_and_save_job_details(job_id, llm1_text, llm3_text, save_path="mydata/output/job_id_output/job_id_output_v7.csv"):
    """
    【修复版】解析并保存：因子名、原始公式、逻辑解释、实现代码
    修复点：正则表达式兼容 (context) 参数，不再强制 (df)。
    """
    
    # 1. 解析 LLM1 (Markdown 表格)
    logic_list = []
    lines = llm1_text.split('\n')
    for line in lines:
        if '|' in line and '因子名称' not in line and '---' not in line:
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 4:
                raw_name = parts[1].replace('**', '').replace('`', '').strip()
                raw_formula = parts[2].replace('`', '').strip()
                logic_desc = parts[3].strip()
                
                if raw_name:
                    logic_list.append({
                        'Factor_Name': raw_name, 
                        'Formula': raw_formula, 
                        'Logic': logic_desc
                    })
    df_logic = pd.DataFrame(logic_list)

    # 2. 解析 LLM3 (Python 代码)
    code_list = []
    chunks = llm3_text.split('def ')
    for chunk in chunks:
        if not chunk.strip(): continue
        
        first_line = chunk.split('\n')[0]
        
        # 🔥🔥🔥 核心修复：放宽正则匹配 🔥🔥🔥
        # 原来: r'calculate_(\w+)\(df\):'  <-- 只能匹配 df
        # 现在: r'calculate_(\w+)\s*\('    <-- 匹配 calculate_Xxx(，不管里面参数叫什么
        match = re.search(r'calculate_(\w+)\s*\(', first_line)
        
        if match:
            factor_name = match.group(1)
            full_code = "def " + chunk.strip()
            code_list.append({'Factor_Name': factor_name, 'Code': full_code})
            
    df_code = pd.DataFrame(code_list)

    # 3. 对齐与合并
    if df_logic.empty or df_code.empty:
        # 增加更详细的调试信息
        print(f"⚠️ Job {job_id} 解析失败: 逻辑表({len(df_logic)}条) | 代码({len(df_code)}条)")
        return

    merged_df = pd.merge(df_logic, df_code, on='Factor_Name', how='inner')
    
    # 4. 生成 ID
    merged_df = merged_df.sort_values('Factor_Name').reset_index(drop=True)
    merged_df['Job_ID'] = job_id

    # 5. 保存
    save_cols = ['Job_ID', 'Factor_Name', 'Formula', 'Logic', 'Code']
    # 防止某些列解析失败导致报错，补全缺失列
    for c in save_cols:
        if c not in merged_df.columns: merged_df[c] = ""
            
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    header = not os.path.exists(save_path)
    
    try:
        merged_df[save_cols].to_csv(save_path, mode='a', header=header, index=False, encoding='utf-8-sig')
        print(f"✅ [索引更新] Job {job_id} 已录入 {len(merged_df)} 个因子 (含公式列)。")
    except Exception as e:
        print(f"❌ 索引保存失败: {e}")

#### pipeline

##### 旧的pipeline（远程）

In [32]:
# import os
# import glob
# import pandas as pd

# def scan_all_fields(data_folder):
#     """
#     扫描指定文件夹下的所有 Parquet 文件名，并按字母顺序打印出来。
#     """
#     if not os.path.exists(data_folder):
#         print(f"❌ 错误：路径不存在 -> {data_folder}")
#         return []
    
#     # 1. 获取所有 parquet 文件
#     files = glob.glob(os.path.join(data_folder, "*.parquet"))
    
#     # 2. 提取纯文件名（去掉路径和后缀）
#     fields = sorted([os.path.splitext(os.path.basename(f))[0] for f in files])
    
#     print(f"✅ 扫描完成！共发现 {len(fields)} 个字段。")
#     print("=" * 50)
    
#     # 3. 打印清单 (方便复制)
#     # 我们按每行 5 个打印，方便查看
#     for i in range(0, len(fields), 5):
#         print(fields[i:i+5])
        
#     print("=" * 50)
#     return fields

# # ==========================================
# # 🚀 请修改下面的路径为你真实的 Parquet 文件夹路径
# # ==========================================
# DATA_PATH = "mydata/cw_data/matrix_data_quarterly2"  # <--- 请确认这里是否正确 (matrix_data 还是 matrix_data2 ?)

# all_fields = scan_all_fields(DATA_PATH)

In [33]:
import json

# 读取JSON文件
with open('财务有效科目_大于1000.json', 'r', encoding='utf-8') as f:
    all_fields = json.load(f)

# 现在data是Python对象（如列表或字典）
# print(all_fields)

In [34]:
len(all_fields)

217

In [35]:
all_fields

['BS_ACCOUNTINGSTANDARDS',
 'BS_ENTERPRISETYPE',
 'BS_CASHEQUIVALENTS',
 'BS_TOTALASSETS',
 'BS_TAXSPAYABLE',
 'BS_TOTALLIABILITY',
 'BS_PAIDINCAPITAL',
 'BS_RETAINEDPROFIT',
 'BS_SEWITHOUTMI',
 'BS_TOTALSHAREHOLDEREQUITY',
 'BS_TOTALLIABILITYANDEQUITY',
 'BS_UPDATETIME',
 'BS_JSID',
 'BS_IFCOMPLETE',
 'BS_TOTALFIXEDASSET',
 'BS_INFOSOURCECODE',
 'BS_INSERTTIME',
 'BS_SALARIESPAYABLE',
 'BS_CAPITALRESERVEFUND',
 'BS_FIXEDASSETS',
 'BS_INTANGIBLEASSETS',
 'BS_SURPLUSRESERVEFUND',
 'BS_DEFERREDTAXASSETS',
 'BS_OTHERPAYABLEED',
 'BS_OTHERRECEIVABLEED',
 'BS_CASH',
 'BS_NOTACCOUNTSPAYABLE',
 'BS_ACCOUNTRECEIVABLE',
 'BS_ACCOUNTSPAYABLE',
 'BS_BILLACCRECEIVABLE',
 'BS_OTHERRECEIVABLE',
 'BS_OTHERPAYABLE',
 'BS_TOTALCURRENTASSETS',
 'BS_TOTALNONCURRENTASSETS',
 'BS_TOTALCURRENTLIABILITY',
 'BS_ADVANCEPAYMENT',
 'BS_OTHERCURRENTASSETS',
 'BS_TOTALNONCURRENTLIABILITY',
 'BS_CONTRACTLIABILITY',
 'BS_INVENTORIES',
 'BS_TCONSTRUINPROCESS',
 'BS_OTHERCURRENTLIABILITY',
 'BS_CONSTRUINPROCESS',
 'BS

In [ ]:
# all_fields = ['name', 'datasource', 'age', 'data_type', 'address', 'data_value']
filtered_fields = list(filter(lambda i: 'DEBT' in i, all_fields))
print(filtered_fields)  # ['datasource', 'data_type', 'data_value']

In [37]:
result = list(set(all_fields) - set(filtered_fields))
# print(result)  # 顺序可能改变，如 [1, 2, 6, 7, 8]
len(result)

217

In [38]:
if 'BS_DATARESOURCESDEVEXPEN' in all_fields:
    print("元素存在")

In [39]:
all_fields = result

#### 新的pipieline

In [40]:
import pandas as pd
import os
import warnings
import re
import uuid
from datetime import datetime

# ==========================================
import pandas as pd
import os
import re
import numpy as np # 确保 numpy 已导入

def parse_code_to_functions(code_text):
    """
    将 LLM 生成的代码文本执行，并提取出以 calculate_ 开头的函数对象。
    【修复版】自动注入 matrix_operators 算子库，解决 NameError。
    """
    clean_code = code_text
    # 清理 Markdown 标记
    if "```python" in code_text:
        match = re.search(r"```python(.*?)```", code_text, re.DOTALL)
        if match: clean_code = match.group(1)
    elif "```" in code_text:
        match = re.search(r"```(.*?)```", code_text, re.DOTALL)
        if match: clean_code = match.group(1)

    local_scope = {}
    try:
        # 1. 获取全局算子库 (确保 Cell 19 已运行)
        if 'matrix_operators' in globals():
            ops = matrix_operators()
        else:
            print("⚠️ 警告: matrix_operators 未定义，算子注入失败！")
            ops = {}

        # 2. 构建全局作用域 (基础库 + 算子库)
        global_scope = {
            'pd': pd, 
            'np': np, 
            'os': os,
            'factor_db': None # 占位防报错
        }
        # 🔥🔥🔥 关键修复：把 YOY, Delay 等注入环境 🔥🔥🔥
        global_scope.update(ops) 

        # 3. 执行代码编译
        exec(clean_code, global_scope, local_scope)
        
    except Exception as e:
        print(f"❌ [Syntax Error] 代码编译失败: {e}")
        return {}

    # 4. 提取函数
    calculators = {}
    for name, func in local_scope.items():
        if name.startswith("calculate_") and callable(func):
            calculators[name] = func
            
    return calculators

In [41]:
import requests
import uuid
import time
import json
import os
import pandas as pd
import warnings
from datetime import datetime

# 远程地址
REMOTE_URL = os.getenv("FACTOR_FACTORY_PLATFORM_URL", "http://localhost:8002")

def run_hybrid_pipeline(
    data_folder, 
    output_folder, 
    useful_fields,
    remote_result_folder,  # <--- 新增参数：专门存放远程 JSON 结果
    input_instruction, 
    client1, client2, 
    max_rounds=1,
    system_prompt = None
):
    warnings.filterwarnings("ignore")
    
    # 1. 初始化
    print(f"🏭 [Init] 初始化因子工厂 (Data: {data_folder})...")
    if 'GLOBAL_TRADING_DAYS' not in globals():
        global_trading_days = pd.date_range('2016-01-01', '2025-12-31', freq='B')
    else:
        global_trading_days = globals()['GLOBAL_TRADING_DAYS']

    db = LazyFactorDB(data_folder, snapshot_days=global_trading_days)
    executor = FactorExecutor(db)
    #可用数据: 这里我们用自己筛选出来的科目进行评估
    all_fields = useful_fields
    print(f"📚 数据库包含 {len(all_fields)} 个可用因子字段")

    for round_i in range(max_rounds):
        print(f"\n🎬 [Hybrid Round {round_i+1}/{max_rounds}] 启动新一轮挖掘...")
        
        # --- Stage 1~3: 智能生成 ---
        history_text = "" 
        if 'load_factor_history' in globals():
            history_text = load_factor_history(max_rounds=10)
        
        print("🧠 [Brain] 正在思考因子逻辑与代码...")
        code_text, design_text = run_three_stages_with_memory(
            client1, client2, 
            history_text=history_text, 
            all_fields=all_fields, 
            extra_instruction=input_instruction
        )
        
        # ==========================================
        # 1️⃣ 第一步：向平台申请 Job ID (Remote Dispatch)
        # ==========================================
        print("\n📡 [Remote] 正在向平台申请 Job ID 并提交任务...")
        
        final_py_code = pack_code_for_remote(OPERATOR_HEADER_STR, code_text)
        session_id = str(uuid.uuid4())
        
        payload = {
            "explain": design_text,
            "py_code": final_py_code,
            "session_id": session_id,
            "exec_type": "cs",
            'use_fundamental':1,
        }
        
        current_job_id = None 
        
        try:
            res = requests.post(f"{REMOTE_URL}/jobs", json=payload)
            if res.status_code == 200:
                res_data = res.json()
                current_job_id = res_data.get('job_id') or res_data.get('id')
                print(f"      ✅ [Remote] 申请成功! Job ID: {current_job_id}")
                
                # 🔥🔥🔥 修复点 1：调用 index_and_save_job_details 🔥🔥🔥
                # 这一步将 JobID 与 LLM 的生成内容（表格 + 代码）永久绑定
                if 'index_and_save_job_details' in globals():
                    print("      📝 [Index] 正在保存因子逻辑索引...")
                    index_and_save_job_details(current_job_id, design_text, code_text)
                else:
                    print("      ⚠️ [Warning] index_and_save_job_details 未定义，跳过索引保存。")

            else:
                print(f"      ❌ [Remote] 申请失败: {res.status_code} - {res.text}")
                
        except Exception as e:
            print(f"      ⚠️ [Remote] 网络通信错误: {e}")

        # ==========================================
        # 2️⃣ 第二步：本地计算与保存 (Local Execution)
        # ==========================================
        print("\n⚙️ [Local] 开始本地计算...")
        
        # 文件夹命名逻辑：优先使用 Job ID，失败则用时间戳兜底
        if current_job_id:
            round_dir_name = str(current_job_id)
        else:
            timestamp_fallback = datetime.now().strftime("%Y%m%d_%H%M%S")
            round_dir_name = f"fallback_{timestamp_fallback}"
            print(f"      ⚠️ [Warning] 使用时间戳兜底保存: {round_dir_name}")

        local_save_dir = os.path.join(output_folder, round_dir_name)
        if not os.path.exists(local_save_dir): os.makedirs(local_save_dir)

        calculators = parse_code_to_functions(code_text)
        if calculators:
            valid_factor_names = [name.replace("calculate_", "") for name in calculators.keys()]
            try:
                results = executor.run(calculators, code_text=code_text)
                
                saved_count = 0
                if results:
                    for factor_name, factor_data in results.items():
                        if factor_name in valid_factor_names:
                            save_path = os.path.join(local_save_dir, f"{factor_name}.parquet")
                            if isinstance(factor_data, pd.DataFrame):
                                factor_data.to_parquet(save_path)
                            elif isinstance(factor_data, pd.Series):
                                factor_data.to_frame(name=factor_name).to_parquet(save_path)
                            print(f"      ✅ [Local] {factor_name} 已保存至 /{round_dir_name}/")
                            saved_count += 1
                    
                    if saved_count == 0:
                        print("      ⚠️ [Local] 计算完成但没有符合规范的因子被保存。")
            except Exception as e:
                print(f"      ❌ [Local] 计算出错: {e}")
                import traceback
                traceback.print_exc()
        else:
            print("      ❌ [Local] 代码解析失败，跳过计算。")

        # ==========================================
        # 3️⃣ 第三步：等待远程结果 (Remote Wait)
        # ==========================================
        if current_job_id:
            print(f"\n⏳ [Remote] 等待平台计算结果 (Job: {current_job_id})...")
            
            # 🔥🔥🔥 修复点 2：使用 remote_result_folder 🔥🔥🔥
            if not os.path.exists(remote_result_folder):
                os.makedirs(remote_result_folder)
            
            # 这里我们把 JSON 文件名也规范化：时间戳_JobID.json
            remote_file_name = f"{datetime.now():%Y%m%d_%H%M%S}_{current_job_id}.json"
            remote_save_path = os.path.join(remote_result_folder, remote_file_name)
            
            wait_and_save_remote_result(
                base_url=REMOTE_URL,
                job_id=current_job_id,
                save_path=remote_save_path
            )

        # ==========================================
        # 收尾
        # ==========================================
        print(f"\n📊 [Summary] 本轮任务 (ID: {current_job_id or 'Unknown'}) 结束。")
        db.clear_cache()
        import gc
        gc.collect()



#### template

In [42]:
# ==========================================
# 1. 定义中金风格策略库 (CICC Logic Library)
# ==========================================
# CICC_STRATEGIES = {
#     "盈利能力 (Profitability)": {
#         "desc": "核心是投入产出效率。请重点挖掘反映【股东回报率】和【利润结构】的因子。",
#         "logic": "1. 杜邦分析拆解：净利率 × 周转率 × 杠杆。\n2. 核心利润占比：扣非净利润/净利润，排除噪音。",
#         "fields": "IS_NETPROFIT (净利), BS_TOTALEQUITY (净资产), IS_OPERATINGREVENUE (营收)"
#     },
#     "成长能力 (Growth)": {
#         "desc": "核心是增长的可持续性。请重点挖掘反映【内生增长】而非并购带来的增长。",
#         "logic": "1. 结合现金流验证增长质量：营收增长的同时，现金流是否同步增长？\n2. 连续性：使用 Delay 函数计算同比(YOY)和环比(QOQ)。",
#         "fields": "IS_REVENUE, IS_NETPROFIT, CFS_NETOPERATECASHFLOW (需计算 YOY)"
#     },
#     "盈余质量 (Earnings Quality)": {
#         "desc": "核心是利润的含金量。请重点挖掘反映【纸面富贵 vs 真金白银】的背离。",
#         "logic": "1. 现金流匹配度：经营性现金流 / 净利润。\n2. 应计项异动：净利润 - 经营现金流（Accruals）。",
#         "fields": "CFS_NETOPERATECASHFLOW, IS_NETPROFIT, BS_ACCOUNTS_RECEIVABLE (应收账款)"
#     },
#     "营运效率 (Operating Efficiency)": {
#         "desc": "核心是资产周转速度。请重点挖掘反映【去库存能力】和【资金占用】的因子。",
#         "logic": "1. 关键周转率：营收 / (期初+期末平均资产)。\n2. 营运周期：存货周转天数 + 应收账款周转天数。",
#         "fields": "IS_OPERATINGCOST, BS_INVENTORY, BS_TOTALASSETS"
#     },
#     "资本结构与安全 (Solvency & Safety)": {
#         "desc": "核心是抗风险能力。请重点挖掘反映【债务压力】和【短期偿付能力】的因子。",
#         "logic": "1. 刚性债务占比：带息债务 / 总投入资本。\n2. 短期流动性：(货币资金+交易性金融资产) / 短期负债。",
#         "fields": "BS_TOTAL_LIABILITIES, BS_TOTALASSETS, BS_CASH_EQUIVALENTS"
#     }
# }

In [43]:
CICC_STRATEGIES = {
    "应计异象与真实盈利 (Accruals & Earnings Truth)": {
        "desc": "核心是利用现金流量表去‘质证’利润表，寻找纸面富贵与真金白银的背离。重点挖掘利润真实度和应计项陷阱。",
        "logic": "1. 净利现金含量：经营现金流净额(CFS_NETOPERATECASHFLOW) / 净利润(IS_NETPROFIT)。\n2. 营收真金白银率：销售商品提供劳务收到的现金(CFS_GOODSSALESERVICERENDERCASH) / 营业收入(IS_OPERATINGREVENUE)。\n3. 资产应计项(反向因子)：(净利润 - 经营现金流净额) / 总资产(BS_TOTALASSETS)。应计项畸高代表极大的业绩下修风险。",
        "fields": "IS_NETPROFIT, CFS_NETOPERATECASHFLOW, IS_OPERATINGREVENUE, CFS_GOODSSALESERVICERENDERCASH, BS_TOTALASSETS, IS_OPERATINGPROFIT"
    },
    "营运剪刀差与压货预警 (Operational Scissors)": {
        "desc": "核心是观察资产负债表与利润表的增速不匹配，挖掘企业是否为了粉饰当期报表而向渠道异常压货。",
        "logic": "1. 应收营收剪刀差(反向因子)：计算应收账款(BS_ACCOUNTRECEIVABLE)的YOY增速减去营业收入的YOY增速。差值越大，造假风险越高。\n2. 存货营收剪刀差(反向因子)：存货(BS_INVENTORIES)YOY增速减去营业收入YOY增速。\n3. 订单蓄水池厚度：(合同负债(BS_CONTRACTLIABILITY) + 预收账款(BS_ADVANCERECEIPTS)) / 营业收入。比例越高，未来业绩确定性越强。",
        "fields": "BS_ACCOUNTRECEIVABLE, IS_OPERATINGREVENUE, BS_INVENTORIES, BS_CONTRACTLIABILITY, BS_ADVANCERECEIPTS"
    },
    "隐性壁垒与研发驱动 (Hidden Moats & R&D)": {
        "desc": "核心是区分‘真正的费用’和‘隐性的资产’。寻找短期利润被研发压制，但长期壁垒深厚的公司。",
        "logic": "1. 研发投入强度：研发费用(IS_RANDD) / 营业收入。\n2. 研发加回核心利润率：(营业利润(IS_OPERATINGPROFIT) + 研发费用) / 营业收入。这能还原科技股最真实的业务盈利能力。\n3. 研发资本化避雷(反向因子)：开发支出(BS_DEVELOPMENTEXPENDITURE) / 研发费用。比例骤增通常是为了强行保住当期利润。",
        "fields": "IS_RANDD, IS_OPERATINGREVENUE, IS_OPERATINGPROFIT, BS_DEVELOPMENTEXPENDITURE"
    },
    "软资产与减值排雷 (Soft Assets Red Flags)": {
        "desc": "核心是度量资产负债表中的‘虚胖’成分。软资产无法变现，且在宏观或行业下行期面临极大的洗大澡(减值)风险。",
        "logic": "1. 软资产占比(反向因子)：(商誉(BS_GOODWILL) + 无形资产(BS_INTANGIBLEASSETS) + 递延所得税资产(BS_DEFERREDTAXASSETS) + 长期待摊费用(BS_LONGDEFERREDEXPENSE)) / 总资产。比例越高越危险。\n2. 隐蔽资金占用(反向因子)：其他应收款(BS_OTHERRECEIVABLE) / 总资产。该值异常通常代表大股东挪用资金。\n3. 减值侵蚀度(反向因子)：(资产减值损失(IS_ASSETIMPAIRMENTLOSS) + 信用减值损失(IS_CREDITIMPAIRMENTP)) / 营业利润。",
        "fields": "BS_GOODWILL, BS_INTANGIBLEASSETS, BS_DEFERREDTAXASSETS, BS_LONGDEFERREDEXPENSE, BS_TOTALASSETS, BS_OTHERRECEIVABLE, IS_ASSETIMPAIRMENTLOSS, IS_CREDITIMPAIRMENTP, IS_OPERATINGPROFIT"
    },
    "自由现金流与资本克制 (FCF & Capital Discipline)": {
        "desc": "核心是衡量企业真实的造血与投资克制能力。在没有外部输血的情况下，企业赚的钱扣除维持性开支后还剩多少。",
        "logic": "1. 纯财务FCF回报率：(经营现金流净额(CFS_NETOPERATECASHFLOW) - 购建固定无形资产支付的现金(CFS_FIXINTANOTHERASSETACQUICASH)) / 总资产。\n2. 资本支出克制率(反向因子)：购建资产支付的现金 / 经营现金流净额。大于1说明公司造血赶不上花钱，处于高耗资危险期。",
        "fields": "CFS_NETOPERATECASHFLOW, CFS_FIXINTANOTHERASSETACQUICASH, BS_TOTALASSETS"
    }
}

In [44]:
# current_prompt

In [45]:
STAGE1_SYSTEM_PROMPT_TEMPLATE = """
你是一位专注于**企业内在价值分析**的资深财务专家，同时也是一名资深量化研究员。
你的目标是基于**纯粹的财务报表数据**，挖掘反映企业经营质量和成长潜力的 Alpha 选股因子。
挖掘的因子必须要有强逻辑支撑。

【🚨 核心禁令】
1. **无价格**：严禁使用 Price, Market Cap, Valuation (PE/PB) 数据。
2. **无行业**：当前环境**没有行业分类数据**。**严禁**设计依赖行业均值/中位数的因子，**严禁**提及行业中性化。

{style_block}

【算子使用规范】
1. **只算原始值**：直接输出计算公式的结果（如 `A / B`）。
2. **全市场排名（可选）**：如果需要标准化，请仅使用 `CS_Rank` (全市场排名)，**不要**使用 `CS_Indus_Rank`。
3. **时序逻辑**：使用 `Delay(d)` 和 `TS_Mean(d)` (d=季度数)。

【输出格式】
1. Markdown 表格：因子名称 | 计算公式 | 逻辑解释
2. 因子名称必须使用英文，不得出现除了下划线以外的特殊符号；
""".strip()

In [46]:
def build_stage1_user_content(history_text: str, data_context_str: str, extra_instruction: str = "") -> str:
    """
    构建 Stage 1 的 User Prompt (纯财务范例版)。
    已修正：范例字段全部替换为真实存在的全大写标准字段，并移除了所有价格/市值依赖。
    """
    guard = ""
    if history_text:
        guard = (
            "【历史回避清单】\n"
            "以下是已生成的因子。请务必创新，**禁止**生成与下方逻辑高度相似的因子：\n"
            f"{history_text.strip()}\n"
        )

#     few_shot_examples = """
# 【编写范例 (纯财务逻辑)】

# ❌ **负面清单 (本轮禁止 - 含价/名字错误)**：
# | 因子名称 | 计算公式 | 错误原因 |
# | :--- | :--- | :--- |
# | **Price_Momentum** | `rank(close / Delay(close, 1))` | **错误**: 用了收盘价 (close)，违反无价格约束。 |
# | **Sales_to_MktCap** | `rank(IS_OPERATINGREVENUE / val_market_cap)` | **错误**: 用了市值 (val_market_cap)，违反无价格约束。 |
# | **Bad_Name_Ratio** | `rank(is_revenue / bs_assets)` | **错误**: 字段名必须全大写且精确匹配白名单，应为 `IS_OPERATINGREVENUE` 和 `BS_TOTALASSETS`。 |
# | **Indus_ROE** | `CS_Indus_Rank(ROE, meta_industry)` | **错误**: 用了行业数据 (meta_industry)，当前环境缺失行业分类，严禁调用。 |

# ✅ **正确示范 (中金研报精选 - 高效Alpha)**：

# | 因子名称 | 计算公式 (标准大写字段) | 逻辑解释 |
# | :--- | :--- | :--- |
# | **CFOA_TTM** | `TTM(CFS_NETOPERATECASHFLOW) / TS_Mean(BS_TOTALASSETS, 3)` | 现金版ROA：中金强推因子。用 TTM 经营现金流除以平均总资产。相比传统 ROA，它剔除了折旧摊销和应计利润的干扰，反映企业每一块钱资产能真正收回多少“真金白银”。 |
# | **GPMD_YOY** | `((IS_OPERATINGREVENUE - IS_OPERATINGCOST)/IS_OPERATINGREVENUE) - Delay((IS_OPERATINGREVENUE - IS_OPERATINGCOST)/IS_OPERATINGREVENUE, 3)` | 营运效率拐点：计算当前单季度毛利率与去年同期的差值。毛利率的同比提升通常意味着产品提价成功或成本控制生效，是业绩爆发的前瞻指标 (Leading Indicator)。 |
# | **Accruals_Ratio** | `(TTM(IS_NETPROFIT) - TTM(CFS_NETOPERATECASHFLOW)) / TS_Mean(BS_TOTALASSETS, 3)` | 盈余质量排雷：计算“净利润”与“经营现金流”的差额占比。差值越大说明利润由大量应收账款堆积而成（水分大），未来股价下跌风险高。该因子数值越低质量越好。 |
# """
    few_shot_examples = """
    # ❌ **负面清单 (本轮禁止 - 含价/名字错误)**：
# | 因子名称 | 计算公式 | 错误原因 |
# | :--- | :--- | :--- |
# | **Price_Momentum** | `rank(close / Delay(close, 1))` | **错误**: 用了收盘价 (close)，违反无价格约束。 |
# | **Sales_to_MktCap** | `rank(IS_OPERATINGREVENUE / val_market_cap)` | **错误**: 用了市值 (val_market_cap)，违反无价格约束。 |
# | **Bad_Name_Ratio** | `rank(is_revenue / bs_assets)` | **错误**: 字段名必须全大写且精确匹配白名单，应为 `IS_OPERATINGREVENUE` 和 `BS_TOTALASSETS`。 |
# | **Indus_ROE** | `CS_Indus_Rank(ROE, meta_industry)` | **错误**: 用了行业数据 (meta_industry)，当前环境缺失行业分类，严禁调用。 |
    
    ✅ **正确示范 (高阶复合财务因子 - 跨表质证与背离)**：

| 因子名称 | 计算公式 (标准大写字段) | 因子方向 | 逻辑解释 |
| :--- | :--- | :--- | :--- |
| **Fund_SUE_YOY** | `Delta(IS_NETPROFIT, 4) / TS_Mean(abs(Delta(IS_NETPROFIT, 4)), 8)` | 1 | 基本面动量加速。计算本季度净利润的同比增量，除以过去两年（8个季度）同比增量的绝对均值。数值越大，说明企业当前利润增长极其罕见、远超自身历史均值，具备强烈的戴维斯双击潜质。 |
| **Recv_Rev_Scissors** | `(BS_ACCOUNTRECEIVABLE - Delay(BS_ACCOUNTRECEIVABLE, 4)) / Delay(BS_ACCOUNTRECEIVABLE, 4) - (IS_OPERATINGREVENUE - Delay(IS_OPERATINGREVENUE, 4)) / Delay(IS_OPERATINGREVENUE, 4)` | -1 | 营运剪刀差预警。计算“应收账款同比增速”减去“营业收入同比增速”。如果该值为正且很大，说明企业为了粉饰当期报表在向渠道异常压货，这种透支未来的增长极不健康。 |
| **RD_Adjusted_Margin** | `(IS_OPERATINGPROFIT + IS_RANDD) / IS_OPERATINGREVENUE` | 1 | 隐性壁垒还原。科技或医药类企业的研发支出（IS_RANDD）在利润表中全额费用化，会严重压制当期营业利润。将研发费用加回营业利润，能剥离财务准则的干扰，还原企业核心业务最真实的赚钱能力。 |
| **Core_Profit_Trend** | `(TTM(IS_OPERATINGPROFIT) - (TTM(IS_NETPROFIT) - TTM(CFS_NETOPERATECASHFLOW))) / BS_TOTALASSETS` | 1 | 高质量内生增长还原。用营业利润减去“应计利润”（净利润与现金流的差额）。剔除掉报表中容易被粉饰的非现金水分，还原出最核心、最可持续的纯粹业务盈利能力。 |

    """

    # 动态注入可用数据字段
    context_part = f"\n{data_context_str}\n"

    task_trigger = """
【本轮任务】
请基于上述【可用数据字段清单】，设计 20 个全新的 **纯基本面因子**。

**数据特性说明**：
当前数据已进行**严格的财报季对齐 (Quarterly Aligned)**。
- 4月30日统一更新年报和一季报。
- 8月30日更新中报。
- 10月30日更新三季报。
因此，使用 `Delay(x, 1)` 可以准确获取上一期财报切片的数据。

**核心约束**：
1. **严格白名单**：必须且只能使用清单中列出的 **全大写** 字段（如 `BS_TOTALASSETS`, `IS_NETPROFIT`）。
2. **算子规范**：
3. **纯财务**：分母严禁使用市值 (`mv`)，请使用总资产 (`BS_TOTALASSETS`)、净资产 (`BS_TOTALSHAREHOLDEREQUITY`) 或 营收 (`IS_OPERATINGREVENUE`) 进行标准化。

请开始输出表格。
"""

    parts = [context_part, guard, few_shot_examples, task_trigger]
    if extra_instruction:
        parts.append(f"【附加指令】\n{extra_instruction}")

    return "\n\n".join([p for p in parts if p])

In [47]:
# 1h

In [48]:
OPERATOR_DOCS = """
# =============================================================================
# 【全能算子库 (Omni-Operator List)】
# 变量说明：X, Y 为数据矩阵 (DataFrame)；d 为时间窗口 (季度数，会自动转换为天数)。
# =============================================================================

# 1. 截面与行业 (Cross-Section)
def CS_Rank(X): 
    '''[标准化] 全市场截面百分比排名 (0~1)。用于去量纲。'''
    pass
def CS_Indus_Rank(X, indus_matrix): 
    '''[核心] 行业中性化排名。
    输入: X(因子), indus_matrix(context['meta_industry'])
    用途: 剔除行业Beta，挖掘纯Alpha。估值/盈利类因子必用。'''
    pass

# 2. 复合关系 (Composite)
def Rank_Div(X, Y): '''GarP策略常用: Rank(X)/Rank(Y)。如: 成长/估值'''
def Rank_Sub(X, Y): '''相对强弱: Rank(X)-Rank(Y)。如: 盈利排名-风险排名'''
def Rank_Mul(X, Y): '''双重确认: Rank(X)*Rank(Y)。'''
def Add(X, Y): '''加法'''
def Sub(X, Y): '''减法 (X-Y)'''
def Mul(X, Y): '''乘法'''
def Div(X, Y): '''除法 (X/Y)'''

# 3. 时序统计 (Time-Series Statistics)
# d=1 代表过去1个季度(约60天)，d=4 代表过去1年(约250天)
def TS_Mean(X, d): '''移动平均线。用于平滑数据，确认长期趋势。'''
def TS_Std(X, d):  '''移动标准差。用于衡量波动率、稳定性。'''
def TS_Sum(X, d):  '''移动求和。'''
def TS_Min(X, d):  '''滚动最小值。用于寻找支撑位或历史底部。'''
def TS_Max(X, d):  '''滚动最大值。用于寻找阻力位或历史顶部。'''
def TS_Corr(X, Y, d): '''滚动相关系数。用于通过量价背离或财务背离挖掘Alpha。'''
def Delta(X, d):   '''时序差分 (X_t - X_{t-d})。用于计算增长额。'''

# 4. 基础变换 (Element-wise)
def TTM(X):  '''[专用] 滚动12个月求和。将单季度财务数据转化为年化数据。'''
def YOY(X):  '''[核心] 同比增长率。((本期-上年同期)/上年同期)。'''
def QOQ(X):  '''环比增长率。'''
def Delay(X, d): '''滞后算子。获取d个季度前的数据。'''
def Log(X):  '''对数。用于市值等偏态数据的正态化。'''
def Abs(X):  '''绝对值。'''
def Sign(X): '''符号函数。'''
def Inv(X):  '''倒数 (1/X)。如将 PE 转换为 EP。'''
"""

In [49]:
# =========================================================================
# Stage 2: 代码生成 (System Prompt - Fundamental Enhanced Edition)
# =========================================================================

# 2. 定义 System Prompt
STAGE2_SYSTEM_PROMPT = f"""
你是一位精通 **Pandas 矩阵运算** 与 **研报复现** 的量化代码工程师。
你的任务是将“基本面因子设计表”中的逻辑，翻译为可执行的 Python 代码。

【核心环境：Lazy Context】
1. **数据源**：所有数据存储在 `context` 字典中。
2. **获取方式**：必须通过 `context['变量名']` 获取 DataFrame。
   - **严禁臆造**：必须严格遵守 Stage 1 提供的【可用数据字段清单】（Strict Whitelist）。
   - **大小写敏感**：财务数据通常为 **全大写**。
     - ✅ 正确：`profit = context['IS_NETPROFIT']` (匹配白名单)
     - ❌ 错误：`profit = context['is_netprofit']` (错误的小写)
     - ❌ 错误：`profit = context['net_profit']` (臆造字段)
   - **行业数据**：`industry = context['meta_industry']` (注意 meta 是小写)
3. **数据结构**：取出的变量均为 (Time x Stock) 的已对齐矩阵，可直接运算。

【核心工具：算子库】
系统已预置了以下全套算子（无需 import，直接调用）。
请**优先使用** `CS_Indus_Rank` 和 `Rank_Div` 等高级算子来提升因子的有效性。

```python
{OPERATOR_DOCS}
【编码规范 (Strict Protocol)】
1. 函数签名：def calculate_FactorName(context):
2. 行业处理：
    严禁 进行行业中性化。
    严禁 使用 CS_Indus_Rank。
    严禁 访问 context['meta_industry']。
    仅允许使用 CS_Rank (全市场排名) 或直接返回原始计算值。
3.复合逻辑：
如果逻辑是“A除以B的相对强弱”，请优先使用 Rank_Div(A, B) 而非简单相除。
4.最终输出：
函数必须返回 factor 变量。
【高分代码范例】 
任务：计算“行业中性化后的 净利润断层 (Profit Surprise)” 逻辑：净利润同比增速 / 行业内排位

def calculate_Revenue_Growth_YOY(context):
    # 1. 提取数据
    revenue = context['IS_OPERATINGREVENUE']
    
    # 2. 计算逻辑
    growth = YOY(revenue)
    
    # 3. 返回原始值 (不做行业处理)
    return growth
""".strip()

In [50]:
### STAGE 2: User Content 构建函数 (`build_stage2_user_content`)

#这一部分基本保持简洁，主要是把 Stage 1 的表格传进去，并做最后的提醒。

def build_stage2_user_content(factor_table_markdown: str) -> str:
    """
    构造 Stage 2 的 User Prompt (基本面适配版)。
    """
    return f"""
下面是上一阶段生成的【纯基本面因子设计表】：

{factor_table_markdown}

请将表中每一个因子翻译为独立的 Python 函数。

**特别提醒**：
1. **Context 访问**：必须使用 `context['完整变量名']` 获取数据。
3. **季度滞后**：涉及到历史财报数据，请务必使用 **`Delay`** (大写)。

请开始生成代码。
""".strip()

In [51]:
# =========================================================================
# Stage 3: 代码修正 (System Prompt - Fundamental Matrix Linter)
# =========================================================================

STAGE3_SYSTEM_PROMPT = """
你是一位资深的量化代码审查员（Linter）。你的任务是检查并修复 Stage 2 生成的代码，确保其符合 **基本面矩阵化工厂** 的运行规范。

【核心运行环境】
1. **数据容器**：不存在 `df` 变量！所有数据必须从 `context` 字典获取。
2. **算子体系**：必须使用系统预置的全局算子（如 `Delay`, `CS_Indus_Rank`），禁止使用 Pandas 原生低效方法。

【⛔️ 严格输出协议 (FATAL ERROR WARNING)】
你的输出将被 Python 的 `exec()` 函数直接执行。
1. **严禁使用 Markdown**：绝对禁止使用 ```python 或 ``` 包裹代码。
2. **严禁任何解释**：不要输出“好的”、“代码如下”或任何文字说明。
3. **纯净模式**：输出必须以 `import` 或 `def` 开头。
4. **后果**：任何非代码字符（包括 Markdown 标记）都会导致系统抛出 `SyntaxError` 并崩溃。

【审查与修复清单 (按优先级执行)】
1. **🔑 字段名合规检查 (Key Validation) - 最高优先级**
   - **检查目标**：检查代码中 `context['KEY']` 的 KEY 是否严格存在于【有效字段白名单】中。
   - **错误现象**：使用小写 (如 `context['bs_assets']`) 或 臆造词 (如 `context['revenue']`)。
   - **强制修正**：必须替换为白名单中对应的 **全大写准确名称** (如 `BS_TOTALASSETS`, `IS_OPERATINGREVENUE`)。
   - **处理无效字段**：如果代码引用了白名单中完全不存在的概念，尝试在白名单中找最接近的替代；若无替代，则将该因子函数标记为无效（注释掉或返回 None）。

2. **🚨 致命错误修正：Context 取数**
   - **错误现象**：使用 `df['...']` 或 `data['...']`。
   - **强制修正**：必须改为 `context['...']`。

3. **⏳ 时间逻辑修正：季度适配**
   - **错误现象**：使用 `shift(1)` 或 `ts_delay(x, 60)`。
   - **强制修正**：改为 `Delay(x, 1)` (1代表1个季度)。
   - **错误现象**：使用 `rolling(250)`。
   - **强制修正**：改为 `TS_Mean(x, 4)` (4代表4个季度)。

4. **🚫 行业去依赖检查**
   - **错误现象**：使用了 `CS_Indus_Rank(x, industry)`。
   - **强制修正**：改为 `CS_Rank(x)` (全市场排名) 或直接返回 `x`。

5. **⚡ 性能与算子规范**
   - **禁止** `groupby`。
   - **禁止** `apply`。

【输出示例】
✅ 正确输出：
import pandas as pd
def calculate_alpha(context):
    return context['BS_TOTALASSETS']

❌ 错误输出 (系统会崩溃)：
```python
import pandas as pd
""".strip()

def build_stage3_user_content(draft_code: str, all_fields: list) -> str:
    """
    构建 Stage 3 User Prompt。
    注意：这里传入的是 all_fields 列表，函数内部将其转为字符串供 Linter 核对。
    """
    # 将列表转为字符串，作为 Linter 的“字典”
    fields_str = str(all_fields)
    
    return f"""
下面是由 Stage 2 生成的代码草稿，请根据系统规范进行严格审查和修复：

{draft_code}

**重点检查列表**：
1. **字段名检查 (Crucial)**：代码中 `context[...]` 里的字段名是否在下方的【有效字段白名单】中？
   - 如果不在（例如是小写），请务必修正为白名单里的大写名称。
   - 如果白名单里完全没有这个字段，请尝试替换或修复。
2. 是否错误地使用了 `df` ? (必须改为 `context`)
3. 是否正确引入了 `meta_industry` 并进行了行业中性化 (`CS_Indus_Rank`)?
4. 滞后操作是否正确使用了 `Delay` (大写)?
5. 是否出现了```python .... ```，有就要将其剔除；
【有效字段白名单 (Valid Field List)】
{fields_str}

请直接输出修复后的代码。
""".strip()

In [52]:
# client_judge_doctor = ChatOpenAI(
#     model="qwen3-max", 
#     api_key=os.getenv("FACTOR_FACTORY_LLM_API_KEY"), 
#     base_url=os.getenv("FACTOR_FACTORY_LLM_BASE_URL", "http://localhost:3000/v1"), 
#     temperature=0,
#     max_retries=5,    # 👈 遇到 503/超时 等报错时，自动重试最多 5 次
#     timeout=120       # 稍微延长超时时间，防止排队太久被掐断
# )

#### client配置

In [53]:
from langchain_openai import ChatOpenAI
client1 = ChatOpenAI(
    # api_key=os.getenv("FACTOR_FACTORY_LLM_API_KEY"),
    api_key = os.getenv("FACTOR_FACTORY_LLM_API_KEY"),
    base_url=os.getenv("FACTOR_FACTORY_LLM_BASE_URL", "http://localhost:3000/v1"),
    model="qwen3-max", # 顺便提醒：请确认qwen3-max是否已发布或拼写正确，目前主流是 qwen-max 或 qwen-plus
    temperature=0,
    streaming=True,
    max_retries=5,
    model_kwargs={
        # 将自定义参数包装在 extra_body 中
        "extra_body": {
            "enable_thinking": True,
        },
        "stream_options": {"include_usage": True}
    }
)


/home/shixi03/.conda/envs/factor/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3519: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [54]:
client2 = ChatOpenAI(
    # 若没有配置环境变量，请在 .env 中设置 FACTOR_FACTORY_LLM_API_KEY
    # api_key=os.getenv("DASHSCOPE_API_KEY"),
    model="qwen-plus",
    # api_key=os.getenv("FACTOR_FACTORY_LLM_API_KEY"),
    api_key = os.getenv("FACTOR_FACTORY_LLM_API_KEY"),
    base_url=os.getenv("FACTOR_FACTORY_LLM_BASE_URL", "http://localhost:3000/v1"),
    temperature=0,
    max_retries=5,
)


#### 循环执行

In [ ]:
# ==========================================
# 🚀 启动脚本 (中金专家轮询版)
# ==========================================

# 1. 基础配置
DATA_PATH = "/storage/server/145server/ly/luodan/财务数据/指标_v0"
OUTPUT_PATH = "mydata/output/factor/日频因子v7"
REMOTE_JSON_PATH = "mydata/output/remote_json"
BASE_INSTRUCTION = "请开始挖掘高alpha的基本面因子" # 基础指令

# 2. 循环执行配置
TOTAL_ROUNDS = 30  # 总共想跑多少轮
rounds = 0         # 计数器
style_keys = list(CICC_STRATEGIES.keys()) # 获取所有风格名称

while True:
    try:
        # --- A. 决定本轮风格 (轮询策略) ---
        # 这里的取余操作 (%) 保证了如果跑30轮，每种风格都会被均匀轮到
        current_style_name = style_keys[rounds % len(style_keys)]
        style_info = CICC_STRATEGIES[current_style_name]
        
        print(f"\n🎬 [Loop {rounds + 1}/{TOTAL_ROUNDS}] 启用专家模式: {current_style_name}")
        
        # --- B. 动态组装 Prompt (注入灵魂) ---
        # 1. 组装 System Prompt (假设您已经按上一步修改了模板)
        style_block_content = f"""
【🎯 本轮专项任务：{current_style_name}】
* **核心逻辑**：{style_info['desc']}
* **分析思路**：{style_info['logic']}
        """
        # 格式化模板 (注意：需要您确保 STAGE1_SYSTEM_PROMPT_TEMPLATE 已被定义)
        # 如果还没定义模板，这里会报错。确保您运行了前面定义的 Cell。
        current_system_prompt = STAGE1_SYSTEM_PROMPT_TEMPLATE.format(
            style_block=style_block_content
        )
        
        # 2. 组装 User Instruction (把推荐字段加在最后)
        # 这样大模型在最后一眼就能看到“小抄”
        current_user_instruction = (
            f"{BASE_INSTRUCTION}\n"
            f"【💡 专家特别提示】为了实现【{current_style_name}】逻辑，"
            f"请优先尝试组合以下字段：\n{style_info['fields']}"
        )

        # --- C. 调用管线 (关键参数修改) ---
        run_hybrid_pipeline(
            data_folder=DATA_PATH,
            output_folder=OUTPUT_PATH,
            useful_fields = all_fields,
            remote_result_folder=REMOTE_JSON_PATH,
            
            # 🔥 关键点1：传入动态生成的指令
            input_instruction=current_user_instruction, 
            
            # 🔥 关键点2：传入动态生成的 System Prompt
            # (前提：您的 run_hybrid_pipeline 函数需要支持这个参数，见下方说明)
            system_prompt=current_system_prompt, 
            
            client1=client1,
            client2=client2,
            
            # 🔥 关键点3：强制设为 1 ！
            # 我们由外层循环控制总轮数，内部每次只跑 1 轮就出来，
            # 这样下一轮循环才能换新的风格。
            max_rounds=1 
        )
        
        # 合并日志
        combine_model_chain_outputs(chain_prefix="model_chain_1_output_")
        combine_model_chain_outputs(chain_prefix="model_chain_3_output_")
        
        print(f"✅ 第 {rounds + 1} 轮 ({current_style_name}) 执行完毕")
        
        # 更新计数与退出
        rounds += 1
        if rounds >= TOTAL_ROUNDS:
            print(f"🎯 达到最大执行轮数 ({TOTAL_ROUNDS})，任务全部完成。")
            break

        # 内存清理
        import gc
        gc.collect()
        time.sleep(5) 

    except Exception as e:
        import traceback
        print(f"🚨 发生未捕获异常: {e}")
        traceback.print_exc()
        print("🔄 正在重试...")
        time.sleep(5)


🎬 [Loop 1/30] 启用专家模式: 应计异象与真实盈利 (Accruals & Earnings Truth)
🏭 [Init] 初始化因子工厂 (Data: /storage/server/145server/ly/luodan/财务数据/指标_v0)...
✅ [DB Init] 宽表引擎初始化: 扫描了 1 个主目录，共索引 544 个科目 (采样点: 30 | 过期阈值: 150天)
📚 数据库包含 217 个可用因子字段

🎬 [Hybrid Round 1/1] 启动新一轮挖掘...
🧠 [Brain] 正在思考因子逻辑与代码...
>>> Calling Stage 1 (Factor Design)...
文件已保存: mydata/output/llm_output/llm_output_v7/model_chain_1_output_20260416_112640.txt
>>> Calling Stage 2 (Code Generation)...
文件已保存: mydata/output/llm_output/llm_output_v7/model_chain_2_output_20260416_112732.txt
>>> Calling Stage 3 (Code Correction)...
文件已保存: mydata/output/llm_output/llm_output_v7/model_chain_3_output_20260416_112811.txt

📡 [Remote] 正在向平台申请 Job ID 并提交任务...
      ✅ [Remote] 申请成功! Job ID: job_dae889ac-a06d-4742-884a-2510d117586f
      📝 [Index] 正在保存因子逻辑索引...
✅ [索引更新] Job job_dae889ac-a06d-4742-884a-2510d117586f 已录入 20 个因子 (含公式列)。

⚙️ [Local] 开始本地计算...
⚙️ [Executor] 启动 V1 模式计算引擎 (Context Dict)...
🕵️ 正在分析算子依赖字段...
   -> 🎯 命中 32 个基础因子: ['BS_DEVELOPMENTEXPEN

Loading Matrices: 100%|██████████| 32/32 [02:03<00:00,  3.86s/it]


✅ Context 字典构建完成，包含 32 个矩阵。
🚀 开始计算 20 个因子 (Matrix Mode)...


Computing: 100%|██████████| 20/20 [00:00<00:00, 95.12it/s]


      ✅ [Local] Accruals_Ratio 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] Accruals_Ratio_QoQ_Delta 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] Core_OperCash_Coverage_Stability 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] Sales_Cash_Ratio_Stability 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] Deferred_Tax_Asset_Change_to_Tax_Expense 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] Deferred_Tax_Liability_Change_to_Tax_Expense 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] Other_Receivable_Growth_Mismatch 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] Contract_Liability_Growth_Mismatch 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] RnD_Capitalization_Rate_QoQ_Delta 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] CIP_Conversion_Ratio 已保存至 /job_dae889ac-a06d-4742-884a-2510d117586f/
      ✅ [Local] Intangible_Amortization_Rate_Rank 已

...

✅ Context 字典构建完成，包含 37 个矩阵。
🚀 开始计算 20 个因子 (Matrix Mode)...


Computing:  30%|███       | 6/20 [00:00<00:00, 34.22it/s]

❌ 因子 Borrowing_Cash_Realization_Efficiency 计算失败: name 'Max' is not defined
❌ 因子 Dividend_Payment_Completion_Ratio 计算失败: name 'Max' is not defined
❌ 因子 Equity_Invest_Cash_Intensity 计算失败: name 'Max' is not defined
❌ 因子 FCF_Asset_Growth_Support_Ratio 计算失败: name 'Max' is not defined
❌ 因子 Soft_Asset_Amortization_Freshness_Index 计算失败: name 'Max' is not defined
❌ 因子 Asset_Disposal_Cash_Recovery_Ratio 计算失败: Add() takes 2 positional arguments but 3 were given


Computing: 100%|██████████| 20/20 [00:00<00:00, 31.65it/s]


      ✅ [Local] Core_OperCash_Cover_OperProfit_Deviation 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] Staff_Tax_Cash_Outflow_Structure_Shift 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] ContractLiab_Inflow_Contribution_Ratio 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] Other_OperCash_Net_Stability_Index 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] Tax_Cash_Expense_Match_Deviation 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] ReceivablesFin_Utilization_Ratio 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] Contract_Asset_Revenue_Conversion_Speed 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] Debt_Investment_Yield_Stability 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] OCF_OperProfit_Coverage_Momentum 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
      ✅ [Local] Advance_Inventory_Structure_Shift 已保存至 /job_99317213-38bd-4b81-996d-13383f1d9a16/
 

In [70]:
url = os.getenv("FACTOR_FACTORY_PLATFORM_URL", "http://localhost:8002")

job_id = 'job_1cfa82fb-ab23-4bb0-a3a3-aba09d21ff2d'

a = requests.get(url + '/jobs/' + job_id)
a.json()

{'job_id': 'job_1cfa82fb-ab23-4bb0-a3a3-aba09d21ff2d',
 'status': 'success',
 'payload': None,
 'results': [{'session_id': 'cdc948d3-23ff-46da-be22-293fe86c49d6',
   'col_name': 'Alpha_VolSkew_CloseOpen_SyncShock_V',
   'ic': -0.0007951605578537967,
   'icir': -0.41984447485033655,
   'rank_ic': -0.005585976726285606,
   'rank_icir': -2.664549856087463,
   'ic_3year': 0.0003192866833039663,
   'icir_3year': 0.1638930616196326,
   'rank_ic_3year': -0.0069348994849767675,
   'rank_icir_3year': -3.109555398354979,
   'group_metics': {'组1': {'sharpe_ratio': 0.37503782435225114,
     'mean_ret': 4.1350446547650936e-05,
     'mdd': 0.05098928367923579},
    '组2': {'sharpe_ratio': 0.8250628203846018,
     'mean_ret': 7.638118694776651e-05,
     'mdd': 0.043148048906126224},
    '组3': {'sharpe_ratio': 0.8535990131533739,
     'mean_ret': 7.176736052354805e-05,
     'mdd': 0.04995664693171947},
    '组4': {'sharpe_ratio': 0.34117701277389895,
     'mean_ret': 2.8971698009694133e-05,
     'mdd': 

#### 远程计算结果核验

In [70]:
url = os.getenv("FACTOR_FACTORY_PLATFORM_URL", "http://localhost:8002")

job_id = 'job_00368217-ca83-448e-a54f-f6b99cebb11e'

a = requests.get(url + '/jobs/' + job_id)
a.json()

{'job_id': 'job_00368217-ca83-448e-a54f-f6b99cebb11e',
 'status': 'success',
 'payload': None,
 'results': [{'session_id': '27e298de-e92e-4f07-b972-0d5aa3c5554f',
   'col_name': 'Net_Profit_to_Average_Equity_ROE_QoQ',
   'ic': -0.0030974312923723536,
   'icir': -0.7743276720948289,
   'rank_ic': 0.00407728983281074,
   'rank_icir': 0.5573419317297079,
   'ic_3year': -0.005674498011371836,
   'icir_3year': -1.4176015600242469,
   'rank_ic_3year': 0.0008828217368232102,
   'rank_icir_3year': 0.09169413414825679,
   'group_metics': {'组1': {'sharpe_ratio': 0.06094149703158534,
     'mean_ret': 1.4637725416360018e-05,
     'mdd': 0.17222810286563844},
    '组2': {'sharpe_ratio': 0.33690699013664543,
     'mean_ret': 6.992563204164723e-05,
     'mdd': 0.20734881321257637},
    '组3': {'sharpe_ratio': 0.4020903696300936,
     'mean_ret': 6.085180449371705e-05,
     'mdd': 0.12258681713080342},
    '组4': {'sharpe_ratio': 0.023305833033853744,
     'mean_ret': 2.949426612573255e-06,
     'mdd': 0